# MAG7 Selective Classifier Final Clean

This clean notebook is ordered as the final dissertation pipeline: setup, data loading, pre-specified horizon handling, feature construction, validation-only horizon audit, frozen-rule refit, held-out test evaluation, ablations, robustness checks, and final summary tables.

The notebook is structurally aligned with the MAG7 full-classifier notebook, but the modelling objective is selective: neutral realised-return rows are created for target audit, then removed before model training and evaluation. The selective model therefore predicts only bullish versus bearish directional outcomes.

**CHANGED in clean notebook:** the original notebook has been reordered and lightly restructured so the final `model_df` is built after the horizon audit. The source notebook is left untouched.

## 1. Aim And Setup

This notebook evaluates a selective MAG7 price-direction classifier. Each retained news event is first labelled as realised bullish, bearish, or neutral, but neutral rows are removed before model training and evaluation. The final selective model therefore classifies directional events as realised bullish or bearish only.

The terminology uses **training**, **validation**, and **test** throughout.

In [1]:
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import binomtest

import matplotlib.pyplot as plt
import seaborn as sns

import ltn

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

# Use the folder the notebook is currently running from.
WORK_DIR = Path.cwd()

# Search only in the current folder for required input files.
EVENTS_PATH = WORK_DIR / "refined_event_returns.parquet"
MAG7_PRICE_PATH = WORK_DIR / "mag7_daily_prices.parquet"
SP500_PRICE_PATH = WORK_DIR / "sp500_daily_prices.parquet"

required_files = {
    "refined_event_returns.parquet": EVENTS_PATH,
    "mag7_daily_prices.parquet": MAG7_PRICE_PATH,
    "sp500_daily_prices.parquet": SP500_PRICE_PATH,
}

missing = [name for name, path in required_files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required file(s) in the current folder:\n"
        + "\n".join(f"- {name}" for name in missing)
        + f"\n\nCurrent folder is:\n{WORK_DIR}"
    )

# SELECTIVE CHANGE:
# Use a separate output folder so selective outputs do not overwrite
# the full-classifier notebook outputs.
OUTPUT_DIR = WORK_DIR / "17_mag7_selective_final_simple_return"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_HORIZON = 3
HORIZON_CANDIDATES = [1, 2, 3, 4, 5]
RETURN_THRESHOLD = 0.005

# SELECTIVE CHANGE:
# Keep the same broad full-classifier chronological split here so the notebooks
# are structurally comparable. The selective difference is in the labels,
# not in the split naming.
TRAINING = ("2023-01-01", "2024-12-31")
VALIDATION = ("2025-01-01", "2025-12-31")
TEST = ("2026-01-01", "2026-06-01")

EPOCHS = 500
LR = 0.01
LOGIC_WEIGHT = 0.30
SEED = 7

ROBUST_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

# SELECTIVE CHANGE:
# The full classifier uses ["bearish", "bullish", "neutral"].
# The selective notebook removes neutral rows before modelling, so the model
# only trains/evaluates bearish vs bullish directional outcomes.
LABELS = ["bearish", "bullish"]

ANTECEDENT_THRESHOLD = 0.50
MIN_TRAINING_EVENTS = 6
MIN_VALIDATION_EVENTS = 5

# SELECTIVE CHANGE:
# Slightly lower training gate matches the earlier selective notebook.
# Validation is kept at 60% to preserve validation-only filtering discipline.
MIN_TRAINING_ACCURACY = 60.0
MIN_VALIDATION_ACCURACY = 60.0

SAVE_OUTPUTS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("working folder:", WORK_DIR)
print("events:", EVENTS_PATH)
print("mag7 prices:", MAG7_PRICE_PATH)
print("sp500 prices:", SP500_PRICE_PATH)
print("output:", OUTPUT_DIR)
print("ltn:", getattr(ltn, "__version__", "unknown"))
print("torch:", torch.__version__, "device:", device)

working folder: /home/jovyan/Stock-Sentiment-Prediction
events: /home/jovyan/Stock-Sentiment-Prediction/refined_event_returns.parquet
mag7 prices: /home/jovyan/Stock-Sentiment-Prediction/mag7_daily_prices.parquet
sp500 prices: /home/jovyan/Stock-Sentiment-Prediction/sp500_daily_prices.parquet
output: /home/jovyan/Stock-Sentiment-Prediction/17_mag7_selective_final_simple_return
ltn: unknown
torch: 2.12.1+cu126 device: cuda


In [2]:
def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

reset_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not force all deterministic algorithms:", e)

## 2. Load Event And Price Data

This section loads the precomputed MAG7 event table, MAG7 prices, and S&P 500 prices. It defines the return-building helpers but does not yet attach a final return horizon to the modelling frame.

For the selective notebook, the realised target is still first constructed as bullish, bearish, or neutral. Neutral rows are removed later, after the horizon-specific target has been attached.

In [3]:
events_all = pd.read_parquet(EVENTS_PATH).copy()
prices = pd.read_parquet(MAG7_PRICE_PATH).copy()
sp500 = pd.read_parquet(SP500_PRICE_PATH).copy()

events_all["time_published"] = pd.to_datetime(events_all["time_published"], utc=True)
events_all["event_date"] = pd.to_datetime(events_all["event_date"])
prices["date"] = pd.to_datetime(prices["date"])
sp500["date"] = pd.to_datetime(sp500["date"])


def build_simple_return_table(price_df: pd.DataFrame, horizons) -> pd.DataFrame:
    frames = []
    for horizon in sorted({int(h) for h in horizons if pd.notna(h)}):
        for ticker, group in price_df.groupby("ticker"):
            g = group.sort_values("date").copy()
            px = pd.to_numeric(g["adj_close"], errors="coerce")
            g["horizon_days"] = horizon
            g["event_date"] = g["date"]
            g["prev_close"] = px.shift(1)
            g["future_close"] = px.shift(-(horizon - 1))
            g["simple_return"] = g["future_close"] / g["prev_close"] - 1.0
            frames.append(g[["ticker", "event_date", "horizon_days", "simple_return", "prev_close", "future_close"]])
    return pd.concat(frames, ignore_index=True)


def realised_label_from_return(value: float, threshold: float = RETURN_THRESHOLD) -> str:
    if pd.isna(value):
        return "neutral"
    if value > threshold:
        return "bullish"
    if value < -threshold:
        return "bearish"
    return "neutral"


# CHANGED in clean notebook:
# Build returns for every candidate horizon before the modelling frame is created.
# The final PRIMARY_HORIZON frame is constructed only after the validation-only horizon audit.
all_return_horizons = sorted(
    set(events_all["horizon_days"].dropna().astype(int).unique())
    | set(HORIZON_CANDIDATES)
    | {PRIMARY_HORIZON}
)
simple_returns = build_simple_return_table(prices, all_return_horizons)

# SELECTIVE CHANGE:
# Keep the event feature frame horizon-independent, exactly like the full classifier.
# Do not remove neutral rows here. Neutral is only removed after a horizon-specific
# realised target has been attached in build_model_df_for_horizon().
events = (
    events_all
    .drop(columns=["simple_return", "prev_close", "future_close", "target_label"], errors="ignore")
    .drop_duplicates("event_id")
    .sort_values(["ticker", "time_published"])
    .reset_index(drop=True)
)

display(events.groupby(events["event_date"].dt.year).size().rename("events").to_frame())

display(events[[
    "ticker", "time_published", "event_date", "representative_title",
    "expected_label", "event_subtype", "valuation_channel", "materiality_strength", "novelty",
]].head())

,events
event_date,
2023,55
2024,122
2025,140
2026,64


,ticker,time_published,event_date,representative_title,expected_label,event_subtype,valuation_channel,materiality_strength,novelty
0,AAPL,2023-07-03 23:40:57+00:00,2023-07-05,"Settlement checks in the mail from Roundup, Ep...",neutral,lawsuit_settlement,legal_liability,medium,material_update
1,AAPL,2023-07-19 22:53:14+00:00,2023-07-20,Report: Apple built ‘Apple GPT’ chatbot using ...,bullish,product_launch,product_adoption,high,new_event
2,AAPL,2023-08-04 06:26:00+00:00,2023-08-04,"Apple sees sales slump continuing, shares drop...",bearish,demand_decrease,revenue_demand,high,material_update
3,AAPL,2023-08-29 00:00:00+00:00,2023-08-29,"Apellis to lay off 25% of staff, trim research...",bearish,layoffs_cost_cutting,cost_margin,high,material_update
4,AAPL,2023-09-22 00:00:00+00:00,2023-09-22,Calix taps Jabil as part of $42B+ broadband ma...,neutral,partnership_customer_deal,unclear,medium,material_update


## 3. Define Return Horizon Before Modelling

`PRIMARY_HORIZON` is declared in setup, and `HORIZON_CANDIDATES` are available before model training. The final labelled modelling frame is intentionally delayed until after the validation-only horizon audit.

For the selective notebook, each candidate horizon is still labelled as bullish, bearish, or neutral first. Neutral rows are then removed inside the horizon-specific modelling frame so that the audit compares bullish/bearish selective classification only.

## 4. Build Horizon-Independent Context Features

This section builds market, ticker, news-flow and Qwen-predicate features. These features are constructed before the final return horizon is attached.

In [4]:
def add_price_context(price_df: pd.DataFrame, prefix: str, group_col: str | None = None) -> pd.DataFrame:
    out = price_df.sort_values(([group_col] if group_col else []) + ["date"]).copy()
    grouped = out.groupby(group_col, group_keys=False) if group_col else [(None, out)]
    
    parts = []
    
    for _, g in grouped:
        g = g.sort_values("date").copy()
        px = pd.to_numeric(g["adj_close"], errors="coerce")
        daily = px.pct_change()
        
        for window in [1, 3, 5, 20]:
            g[f"{prefix}_ret_{window}d_pre"] = px.pct_change(window).shift(1)
        
        g[f"{prefix}_abs_ret_1d_pre"] = daily.abs().shift(1)
        g[f"{prefix}_vol_5d_pre"] = daily.rolling(5).std().shift(1)
        g[f"{prefix}_vol_20d_pre"] = daily.rolling(20).std().shift(1)
        
        parts.append(g)
        
    return pd.concat(parts, ignore_index=True)

ticker_ctx = add_price_context(prices, "ticker", "ticker")
sp500_ctx = add_price_context(sp500, "sp500", None).drop(columns=["ticker"], errors="ignore")

ctx = events.merge(
    ticker_ctx[[
        "ticker", "date", "ticker_ret_1d_pre", "ticker_ret_3d_pre", "ticker_ret_5d_pre", "ticker_ret_20d_pre",
        "ticker_abs_ret_1d_pre", "ticker_vol_5d_pre", "ticker_vol_20d_pre",
    ]],
    left_on=["ticker", "event_date"],
    right_on=["ticker", "date"],
    how="left",
).drop(columns=["date"], errors="ignore")

ctx = ctx.merge(
    sp500_ctx[[
        "date", "sp500_ret_1d_pre", "sp500_ret_3d_pre", "sp500_ret_5d_pre", "sp500_ret_20d_pre",
        "sp500_abs_ret_1d_pre", "sp500_vol_5d_pre", "sp500_vol_20d_pre",
    ]],
    left_on="event_date",
    right_on="date",
    how="left",
).drop(columns=["date"], errors="ignore")


for h in ["1d", "3d", "5d", "20d"]:
    ctx[f"relative_ret_{h}_pre"] = ctx[f"ticker_ret_{h}_pre"] - ctx[f"sp500_ret_{h}_pre"]

ctx = ctx.sort_values(["ticker", "time_published"]).reset_index(drop=True)
ctx["event_density_ticker_7d"] = 0.0
ctx["event_density_ticker_30d"] = 0.0
ctx["similar_density_ticker_30d"] = 0.0

for ticker, group in ctx.groupby("ticker"):
    for idx, ts, subtype, channel in zip(
        group.index,
        group["time_published"],
        group["event_subtype"],
        group["valuation_channel"],
    ):
        prev = group.loc[group["time_published"].lt(ts)]
        last7 = prev.loc[prev["time_published"].ge(ts - pd.Timedelta(days=7))]
        last30 = prev.loc[prev["time_published"].ge(ts - pd.Timedelta(days=30))]
        similar30 = last30.loc[
            last30["event_subtype"].eq(subtype)
            | last30["valuation_channel"].eq(channel)
        ]

        ctx.loc[idx, "event_density_ticker_7d"] = len(last7)
        ctx.loc[idx, "event_density_ticker_30d"] = len(last30)
        ctx.loc[idx, "similar_density_ticker_30d"] = len(similar30)


hour = ctx["time_published"].dt.hour + ctx["time_published"].dt.minute / 60
ctx["hour_sin_01"] = (np.sin(2 * np.pi * hour / 24) + 1) / 2
ctx["hour_cos_01"] = (np.cos(2 * np.pi * hour / 24) + 1) / 2
ctx["is_monday"] = (ctx["time_published"].dt.weekday == 0).astype(float)
ctx["is_friday"] = (ctx["time_published"].dt.weekday == 4).astype(float)

display(ctx[[
    "ticker", "event_date", "expected_label",
    "relative_ret_5d_pre", "sp500_vol_5d_pre", "ticker_vol_5d_pre",
    "event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d",
    "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
]].head(10))

,ticker,event_date,expected_label,relative_ret_5d_pre,sp500_vol_5d_pre,ticker_vol_5d_pre,event_density_ticker_7d,event_density_ticker_30d,similar_density_ticker_30d,hour_sin_01,hour_cos_01,is_monday,is_friday
0,AAPL,2023-07-05,neutral,0.009523,0.005807,0.011915,0.0,0.0,0.0,0.456422,0.998097,1.0,0.0
1,AAPL,2023-07-20,bullish,0.007166,0.003793,0.007295,0.0,1.0,0.0,0.355902,0.978786,0.0,0.0
2,AAPL,2023-08-04,bearish,-0.002781,0.008558,0.010984,0.0,1.0,0.0,0.996786,0.443398,0.0,1.0
3,AAPL,2023-08-29,bearish,0.017115,0.009786,0.018306,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0
4,AAPL,2023-09-22,neutral,0.028568,0.007071,0.014146,0.0,1.0,0.0,0.500000,1.000000,0.0,1.0
5,AAPL,2023-10-30,bullish,-0.001674,0.008593,0.013354,0.0,0.0,0.0,0.402455,0.009607,0.0,0.0
6,AAPL,2023-11-14,bearish,0.021998,0.008614,0.012786,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0
7,AAPL,2023-11-30,bullish,-0.009392,0.002293,0.004825,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0
8,AAPL,2023-12-19,bearish,-0.011525,0.005160,0.009768,0.0,1.0,0.0,0.441231,0.996534,1.0,0.0
9,AAPL,2024-01-10,bearish,-0.005576,0.008351,0.014335,0.0,1.0,0.0,0.500000,1.000000,0.0,0.0


## 5. Create Chronological Training / Validation / Test Split

This section creates the chronological split and feature specifications. It builds `feature_df`, which is horizon-independent: it contains features and period labels, but not the final `simple_return` or `target_label`.

For the selective notebook, the split is created before neutral rows are removed. The bullish/bearish filtering happens later inside the horizon-specific modelling frame, after each candidate horizon has been labelled.

In [5]:
def assign_period(d):
    d = pd.Timestamp(d)
    if pd.Timestamp(TRAINING[0]) <= d <= pd.Timestamp(TRAINING[1]):
        return "training"
    if pd.Timestamp(VALIDATION[0]) <= d <= pd.Timestamp(VALIDATION[1]):
        return "validation"
    if pd.Timestamp(TEST[0]) <= d <= pd.Timestamp(TEST[1]):
        return "test"
    return "outside"

ctx["period"] = ctx["event_date"].map(assign_period)

fit_mask = ctx["period"].eq("training")

def fit_abs_scale(col, q=0.80):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    scale = s.abs().quantile(q)
    return float(scale) if np.isfinite(scale) and scale != 0 else 1.0

def fit_quantile(col, q, fallback=1.0):
    s = pd.to_numeric(ctx.loc[fit_mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    value = s.quantile(q)
    return float(value) if np.isfinite(value) and value != 0 else fallback

def soft_pos_with_scale(s, scale):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s.clip(lower=0) / scale).clip(0, 1).fillna(0)

# SELECTIVE CLEANUP:
# The pasted full-classifier cell had a duplicate broken soft_neg_with_scale(s)
# definition that referenced an undefined variable `scale`. Keep only this
# correct two-argument version.
def soft_neg_with_scale(s, scale):
    return soft_pos_with_scale(-pd.to_numeric(s, errors="coerce"), scale)

def soft_low_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (1 - (s / cutoff)).clip(0, 1).fillna(0)

def soft_high_with_cutoff(s, cutoff):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return (s / cutoff - 1).clip(0, 1).fillna(0)

ticker_5d_scale = fit_abs_scale("ticker_ret_5d_pre")
market_5d_scale = fit_abs_scale("sp500_ret_5d_pre")
relative_5d_scale = fit_abs_scale("relative_ret_5d_pre")

ctx["ticker_prior_up_5d"] = soft_pos_with_scale(ctx["ticker_ret_5d_pre"], ticker_5d_scale)
ctx["ticker_prior_down_5d"] = soft_neg_with_scale(ctx["ticker_ret_5d_pre"], ticker_5d_scale)
ctx["market_prior_up_5d"] = soft_pos_with_scale(ctx["sp500_ret_5d_pre"], market_5d_scale)
ctx["market_prior_down_5d"] = soft_neg_with_scale(ctx["sp500_ret_5d_pre"], market_5d_scale)
ctx["relative_prior_up_5d"] = soft_pos_with_scale(ctx["relative_ret_5d_pre"], relative_5d_scale)
ctx["relative_prior_down_5d"] = soft_neg_with_scale(ctx["relative_ret_5d_pre"], relative_5d_scale)

ctx["quiet_market_5d"] = soft_low_with_cutoff(
    ctx["sp500_vol_5d_pre"],
    fit_quantile("sp500_vol_5d_pre", 0.40),
)
ctx["volatile_market_5d"] = soft_high_with_cutoff(
    ctx["sp500_vol_5d_pre"],
    fit_quantile("sp500_vol_5d_pre", 0.70),
)
ctx["quiet_ticker_5d"] = soft_low_with_cutoff(
    ctx["ticker_vol_5d_pre"],
    fit_quantile("ticker_vol_5d_pre", 0.40),
)
ctx["volatile_ticker_5d"] = soft_high_with_cutoff(
    ctx["ticker_vol_5d_pre"],
    fit_quantile("ticker_vol_5d_pre", 0.70),
)

for col in ["event_density_ticker_7d", "event_density_ticker_30d", "similar_density_ticker_30d"]:
    denom = max(ctx.loc[fit_mask, col].quantile(0.95), 1)
    ctx[col + "_01"] = (ctx[col] / denom).clip(0, 1)

ctx["novel_context_30d"] = (1 - ctx["similar_density_ticker_30d_01"]).clip(0, 1)

dummy_parts = []
dummy_cols = []

categorical_cols = [
    "expected_label",
    "event_subtype",
    "valuation_channel",
    "materiality_strength",
    "novelty",
]

for col in categorical_cols:
    categories = sorted(set(ctx.loc[fit_mask, col].fillna("missing").astype(str)))
    values = ctx[col].fillna("missing").astype(str)

    for category in categories:
        dummy_name = f"{col}_{category}"
        dummy_parts.append(values.eq(category).astype(float).rename(dummy_name))
        dummy_cols.append(dummy_name)

dummies = pd.concat(dummy_parts, axis=1)
ctx = pd.concat([ctx, dummies], axis=1)

context_cols = [
    "ticker_prior_up_5d", "ticker_prior_down_5d", "market_prior_up_5d", "market_prior_down_5d",
    "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
    "quiet_ticker_5d", "volatile_ticker_5d",
    "event_density_ticker_7d_01", "event_density_ticker_30d_01", "similar_density_ticker_30d_01",
    "novel_context_30d", "hour_sin_01", "hour_cos_01", "is_monday", "is_friday",
]

qwen_cols = list(dummies.columns) + ["include_candidate", "article_count"]
for col in ["include_candidate", "article_count"]:
    ctx[col] = pd.to_numeric(ctx[col], errors="coerce").fillna(0)

article_count_denom = max(ctx.loc[fit_mask, "article_count"].quantile(0.95), 1)
ctx["article_count"] = (ctx["article_count"] / article_count_denom).clip(0, 1)

# Interaction features: Qwen event predicates multiplied by context truth values.
interaction_qwen_cols = [
    c for c in dummy_cols
    if c.startswith("expected_label_")
    or c.startswith("valuation_channel_")
    or c.startswith("event_subtype_")
    or c.startswith("materiality_strength_")
]

interaction_context_cols = [
    "relative_prior_up_5d", "relative_prior_down_5d", "quiet_market_5d", "volatile_market_5d",
    "quiet_ticker_5d", "volatile_ticker_5d", "event_density_ticker_7d_01", "novel_context_30d",
]

interaction_data = {}
for q in interaction_qwen_cols:
    for c in interaction_context_cols:
        name = f"interaction__{q}__x__{c}"
        interaction_data[name] = ctx[q].fillna(0) * ctx[c].fillna(0)

interaction_cols = list(interaction_data.keys())
if interaction_data:
    ctx = pd.concat([ctx, pd.DataFrame(interaction_data, index=ctx.index)], axis=1)

# CHANGED in clean notebook:
# This frame contains event predicates, context features, interactions, and the chronological split only.
# Future returns and realised labels are merged in after the horizon audit.
#
# SELECTIVE CHANGE:
# Do not remove neutral rows here. `feature_df` must stay horizon-independent.
# Neutral rows are removed later after each candidate horizon is labelled.
feature_df = ctx.loc[ctx["period"].isin(["training", "validation", "test"])].copy()

for col in qwen_cols + context_cols + interaction_cols:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

rule_block_feature_cols = {
    "qwen_semantic": list(qwen_cols),
    "market_context": list(context_cols),
    "qwen_context_interaction": list(interaction_cols),
}

display(feature_df["period"].value_counts().rename("events").to_frame())

# SELECTIVE CHANGE:
# Same four structural variants as the full-classifier notebook, but renamed
# as selective variants. The model will later train only on LABELS =
# ["bearish", "bullish"].
variant_specs = {
    "selective_ltn_qwen_only": {
        "feature_cols": qwen_cols,
        "rule_blocks": ["qwen_semantic"],
    },
    "selective_ltn_context_only": {
        "feature_cols": context_cols,
        "rule_blocks": ["market_context"],
    },
    "selective_ltn_qwen_plus_context": {
        "feature_cols": qwen_cols + context_cols,
        "rule_blocks": ["qwen_semantic", "market_context"],
    },
    "selective_ltn_qwen_context_interaction": {
        "feature_cols": qwen_cols + context_cols + interaction_cols,
        "rule_blocks": ["qwen_semantic", "market_context", "qwen_context_interaction"],
    },
}

for spec in variant_specs.values():
    spec["feature_cols"] = list(dict.fromkeys(spec["feature_cols"]))
    spec["return_col"] = "simple_return"

feature_manifest = pd.DataFrame([
    {
        "variant": name,
        "n_features": len(spec["feature_cols"]),
        "rule_blocks": ", ".join(spec["rule_blocks"]),
        "features": ", ".join(spec["feature_cols"]),
    }
    for name, spec in variant_specs.items()
])

if SAVE_OUTPUTS:
    feature_manifest.to_csv(OUTPUT_DIR / "selective_ltn_feature_manifest.csv", index=False)

display(feature_manifest)

,events
period,
training,177
validation,140
test,64


,variant,n_features,rule_blocks,features
0,selective_ltn_qwen_only,46,qwen_semantic,"expected_label_bearish, expected_label_bullish..."
1,selective_ltn_context_only,18,market_context,"ticker_prior_up_5d, ticker_prior_down_5d, mark..."
2,selective_ltn_qwen_plus_context,64,"qwen_semantic, market_context","expected_label_bearish, expected_label_bullish..."
3,selective_ltn_qwen_context_interaction,384,"qwen_semantic, market_context, qwen_context_in...","expected_label_bearish, expected_label_bullish..."


## 6. Define Model Variants And LTN Helpers

This section defines the LTN predicates, rule-mining utilities, model-training routine, scoring helper, and summary metrics.

For the selective notebook, the helper structure is kept aligned with the full-classifier notebook, but the model defines only bullish and bearish predicates. Neutral rows are removed before training and are not modelled as a third class.

In [6]:
class OutcomeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = max(12, min(80, input_dim * 2))
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).reshape(-1)

def truth_value(obj):
    return obj.value if hasattr(obj, "value") else obj

def feature_tensor(data, cols):
    x = data[cols].replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1).to_numpy(dtype=np.float32)
    return torch.tensor(x, dtype=torch.float32, device=device)

def target_tensors(data):
    # SELECTIVE CHANGE:
    # Full classifier returns bearish, bullish, and neutral tensors.
    # Selective notebook uses LABELS = ["bearish", "bullish"], so this returns
    # only directional tensors and cannot create a neutral target.
    y = pd.get_dummies(data["target_label"]).reindex(columns=LABELS, fill_value=0).to_numpy(dtype=np.float32)
    return {
        label: torch.tensor(y[:, i].reshape(-1, 1), dtype=torch.float32, device=device)
        for i, label in enumerate(LABELS)
    }

def label_return_alignment(data, label, return_col):
    ret = pd.to_numeric(data[return_col], errors="coerce")
    if label == "bullish":
        signed = ret
    elif label == "bearish":
        signed = -ret
    else:
        raise ValueError(f"Selective notebook only supports bullish/bearish labels, got: {label}")

    signed = signed.dropna()
    if len(signed) < 3:
        return np.nan, np.nan

    # SELECTIVE CHANGE:
    # No neutral branch. Directional labels use the same one-sided sign test.
    p = binomtest(int((signed > 0).sum()), len(signed), 0.5, alternative="greater").pvalue

    return float(100 * signed.mean()), float(p) if np.isfinite(p) else np.nan

def empirical_rule_stats(data, condition_cols, label, return_col):
    if len(data) == 0:
        return {"n": 0, "accuracy_pct": np.nan, "aligned_mean_pct": np.nan, "aligned_signflip_p": np.nan}
    active = np.ones(len(data), dtype=bool)
    for col in condition_cols:
        active &= pd.to_numeric(data[col], errors="coerce").fillna(0).to_numpy() >= ANTECEDENT_THRESHOLD
    subset = data.loc[active].copy()
    if subset.empty:
        return {"n": 0, "accuracy_pct": np.nan, "aligned_mean_pct": np.nan, "aligned_signflip_p": np.nan}
    aligned_mean, p = label_return_alignment(subset, label, return_col)
    return {
        "n": int(len(subset)),
        "accuracy_pct": float(100 * subset["target_label"].eq(label).mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": p,
    }

def infer_rule_block(feature_name):
    # SELECTIVE CLEANUP:
    # Interaction features in this notebook are named interaction__, not gate__.
    if feature_name.startswith("interaction__"):
        return "qwen_context_interaction"

    if feature_name in context_cols:
        return "market_context"

    if feature_name in qwen_cols:
        return "qwen_semantic"

    return "unknown"

def mine_validated_rules(
    data,
    feature_cols,
    return_col,
    rule_blocks=None,
    max_pair_features=36,
    max_rules_per_label=12,
):
    training_df = data.loc[data["period"].eq("training")].copy()
    validation_df = data.loc[data["period"].eq("validation")].copy()

    cols = [c for c in feature_cols if c in data.columns]

    if rule_blocks is not None:
        allowed_blocks = set(rule_blocks)
        cols = [c for c in cols if infer_rule_block(c) in allowed_blocks]

    active_counts = training_df[cols].ge(ANTECEDENT_THRESHOLD).sum().sort_values(ascending=False)
    pair_base = active_counts.head(max_pair_features).index.tolist()

    candidates = [(c,) for c in cols] + list(combinations(pair_base, 2))

    rows = []

    for condition_cols in candidates:
        condition_cols = tuple(condition_cols)

        for label in LABELS:
            disc = empirical_rule_stats(training_df, condition_cols, label, return_col)

            if disc["n"] < MIN_TRAINING_EVENTS or not np.isfinite(disc["accuracy_pct"]):
                continue

            val = empirical_rule_stats(validation_df, condition_cols, label, return_col)

            if val["n"] < MIN_VALIDATION_EVENTS or not np.isfinite(val["accuracy_pct"]):
                continue

            score = (
                disc["accuracy_pct"] / 100
                + val["accuracy_pct"] / 100
                + min(disc["n"] / 40, 1)
                + min(val["n"] / 30, 1)
            )

            # SELECTIVE CHANGE:
            # No neutral-specific scoring branch. All candidate rules are
            # directional bullish/bearish rules, so positive aligned return is rewarded.
            if np.isfinite(disc["aligned_mean_pct"]):
                score += min(max(disc["aligned_mean_pct"], 0) / 2, 1)

            rows.append({
                "condition": " & ".join(condition_cols),
                "condition_cols_json": json.dumps(condition_cols),
                "rule_label": label,
                "rule_score": score,
                # CHANGED in clean notebook:
                # Store which rule blocks are involved, so the selected rules are auditable.
                "rule_blocks": ", ".join(sorted({infer_rule_block(c) for c in condition_cols})),
                **{f"training_{k}": v for k, v in disc.items()},
                **{f"validation_{k}": v for k, v in val.items()},
            })

    candidates_df = pd.DataFrame(rows)

    if candidates_df.empty:
        return candidates_df, candidates_df

    selected = candidates_df.loc[
        candidates_df["training_accuracy_pct"].ge(MIN_TRAINING_ACCURACY)
        & candidates_df["validation_accuracy_pct"].ge(MIN_VALIDATION_ACCURACY)
    ].copy()

    selected = selected.sort_values(
        ["rule_label", "validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=[True, False, False, False],
    )

    selected = selected.groupby("rule_label", group_keys=False).head(max_rules_per_label)

    selected = selected.sort_values(
        ["validation_accuracy_pct", "validation_n", "rule_score"],
        ascending=False,
    ).reset_index(drop=True)

    return (
        candidates_df.sort_values("rule_score", ascending=False).reset_index(drop=True),
        selected,
    )

class FeaturePredicate(nn.Module):
    def __init__(self, index):
        super().__init__()
        self.index = index

    def forward(self, z):
        return z[:, self.index].reshape(-1, 1)

def make_feature_predicates(feature_cols):
    preds = {}
    for i, col in enumerate(feature_cols):
        preds[col] = ltn.Predicate(model=FeaturePredicate(i).to(device))
    return preds

def antecedent_from_rule(cols, feat_preds, x, AndOp):
    expr = truth_value(feat_preds[cols[0]](x))
    for col in cols[1:]:
        expr = AndOp(expr, truth_value(feat_preds[col](x)))
    return expr

def train_variant(name, spec, train_df, use_logic=True, logic_weight=LOGIC_WEIGHT, seed=SEED, mine_rules=True):
    reset_seed(seed)
    
    feature_cols = spec["feature_cols"]
    return_col = spec["return_col"]

    # mine fresh rules (training)
    if use_logic and mine_rules:
        candidates, selected_rules = mine_validated_rules(
            train_df,
            feature_cols,
            return_col,
            rule_blocks=spec.get("rule_blocks"),
            max_pair_features=spec.get("max_pair_features", 36),
            max_rules_per_label=spec.get("max_rules_per_label", 12),
        )

    # use frozen rules (after validation)
    elif use_logic and not mine_rules:
        candidates = pd.DataFrame()

        if "selected_rules" not in spec:
            raise ValueError(
                f"{name} was called with mine_rules=False, "
                "but spec['selected_rules'] was not provided."
            )

        selected_rules = spec["selected_rules"].copy()
        
    else:
        candidates = pd.DataFrame()
        selected_rules = pd.DataFrame()
    
    if SAVE_OUTPUTS and use_logic:
        candidates.to_csv(OUTPUT_DIR / f"{name}_candidate_mined_rules.csv", index=False)
        selected_rules.to_csv(OUTPUT_DIR / f"{name}_selected_validated_rules.csv", index=False)

    X = feature_tensor(train_df, feature_cols)
    x = ltn.Variable(f"x_{name}", X)
    y = target_tensors(train_df)
    feat_preds = make_feature_predicates(feature_cols)

    Bearish = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))
    Bullish = ltn.Predicate(model=OutcomeMLP(X.shape[1]).to(device))

    # SELECTIVE CHANGE:
    # Full classifier creates Bearish, Bullish, and Neutral predicates.
    # Selective notebook creates only Bearish and Bullish predicates.
    AndOp = ltn.fuzzy_ops.AndProd()
    ImpliesOp = ltn.fuzzy_ops.ImpliesReichenbach()
    ForallAgg = ltn.fuzzy_ops.AggregPMeanError(p=2)
    SatAgg = ltn.fuzzy_ops.SatAgg()

    label_predicates = {
        "bearish": Bearish,
        "bullish": Bullish,
    }
    
    rule_specs = []
    for _, rule in selected_rules.iterrows():
        cols = [c for c in json.loads(rule["condition_cols_json"]) if c in feat_preds]
        if cols:
            rule_specs.append((rule["condition"], cols, rule["rule_label"]))

    params = (
        list(Bearish.model.parameters())
        + list(Bullish.model.parameters())
    )

    opt = torch.optim.Adam(params, lr=LR)
    
    history = []
    for epoch in range(EPOCHS):
        opt.zero_grad()
        
        pred_bear = truth_value(Bearish(x)).reshape(-1, 1)
        pred_bull = truth_value(Bullish(x)).reshape(-1, 1)
        
        supervised = (
            F.mse_loss(pred_bear, y["bearish"])
            + F.mse_loss(pred_bull, y["bullish"])
        )
        
        # SELECTIVE CHANGE:
        # Exclusivity only penalises simultaneous bearish and bullish truth.
        # There is no neutral predicate in this notebook.
        exclusivity = torch.mean(pred_bear * pred_bull)
        
        formulas = []
        if use_logic:            
            for rule_name, cols, label in rule_specs:
                formulas.append((
                    rule_name,
                    ForallAgg(
                        ImpliesOp(
                            antecedent_from_rule(cols, feat_preds, x, AndOp),
                            truth_value(label_predicates[label](x)),
                        )
                    ),
                ))
                
        sat = SatAgg(*[formula for _, formula in formulas]) if formulas else torch.tensor(1.0, device=device)
        
        logic_penalty = logic_weight * (1.0 - truth_value(sat)) if use_logic else torch.tensor(0.0, device=device)
        
        loss = supervised + 0.10 * exclusivity + logic_penalty
        loss.backward()
        opt.step()
        
        if epoch % 50 == 0 or epoch == EPOCHS - 1:
            row = {
                "variant": name,
                "epoch": epoch,
                "seed": seed,
                "use_logic": use_logic,
                "mine_rules": mine_rules,
                "logic_weight": logic_weight,
                "loss": float(loss.detach().cpu()),
                "supervised_mse": float(supervised.detach().cpu()),
                "exclusivity": float(exclusivity.detach().cpu()),
                "sat": float(truth_value(sat).detach().cpu()) if hasattr(truth_value(sat), "detach") else float(sat),
                "n_selected_rules": len(rule_specs),
            }
            
            for rule_name, formula in formulas[:20]:
                row[f"sat_{rule_name[:80]}"] = float(truth_value(formula).detach().cpu())
                
            history.append(row)

    return {
        "name": name,
        "seed": seed,
        "feature_cols": feature_cols,
        "candidate_rules": candidates,
        "selected_rules": selected_rules,
        "use_logic": use_logic,
        "mine_rules": mine_rules,
        "logic_weight": logic_weight,
        "Bearish": Bearish,
        "Bullish": Bullish,
        # SELECTIVE CHANGE:
        # No "Neutral" object is returned.
        "history": pd.DataFrame(history),
    }

In [7]:
def score_variant(bundle, df):
    out = df.copy()
    X = feature_tensor(out, bundle["feature_cols"])

    with torch.no_grad():
        eval_x = ltn.Variable(f"eval_{bundle['name']}", X)
        out["score_bearish"] = truth_value(bundle["Bearish"](eval_x)).detach().cpu().numpy().reshape(-1)
        out["score_bullish"] = truth_value(bundle["Bullish"](eval_x)).detach().cpu().numpy().reshape(-1)

        # SELECTIVE CHANGE:
        # Full classifier also scores bundle["Neutral"] and creates score_neutral.
        # Selective notebook has no Neutral predicate, so only bearish/bullish scores exist.

    scores = out[["score_bearish", "score_bullish"]].to_numpy()

    # SELECTIVE CHANGE:
    # Prediction is argmax over LABELS = ["bearish", "bullish"] only.
    out["prediction"] = np.array(LABELS)[scores.argmax(axis=1)]

    # With two classes, this is simply the gap between bullish and bearish scores.
    # Keeping the same column name preserves the full-classifier table structure.
    out["confidence_margin"] = np.abs(out["score_bullish"] - out["score_bearish"])

    return out

def aligned_stats(df):
    ret = pd.to_numeric(df["simple_return"], errors="coerce")
    pred = df["prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )

    signed = pd.Series(signed, index=df.index).dropna()

    if len(signed) < 3:
        return np.nan, np.nan

    p = binomtest(int((signed > 0).sum()), len(signed), 0.5, alternative="greater").pvalue

    return float(signed.mean() * 100), float(p)

def summary_row(variant, period, df, bundle=None):
    aligned_mean, aligned_p = aligned_stats(df)

    row = {
        "variant": variant,
        "period": period,
        "n": int(len(df)),
        "accuracy_pct": float(100 * df["prediction"].eq(df["target_label"]).mean()),
        "mean_simple_return_pct": float(100 * pd.to_numeric(df["simple_return"], errors="coerce").mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": aligned_p,
        "bullish_pred_rate_pct": float(100 * df["prediction"].eq("bullish").mean()),
        "bearish_pred_rate_pct": float(100 * df["prediction"].eq("bearish").mean()),

        # SELECTIVE CHANGE:
        # Full classifier has neutral_pred_rate_pct.
        # Selective notebook removes neutral rows and does not predict neutral,
        # so this column is intentionally omitted.
        "mean_confidence_margin": float(df["confidence_margin"].mean()),
    }

    if bundle is not None:
        row["use_logic"] = bundle["use_logic"]
        row["logic_weight"] = bundle["logic_weight"]
        row["n_selected_rules"] = len(bundle["selected_rules"])

    return row

## 7. Validation-Only Horizon Audit

This audit compares candidate return horizons using training and validation data only. The held-out test period is excluded from this step.

**CHANGED in clean notebook:** the audit now builds each horizon-specific modelling frame from `feature_df`, so the final `PRIMARY_HORIZON` frame is not created until after this audit.

For the selective notebook, each candidate horizon is first labelled as bullish, bearish, or neutral. Neutral rows are then removed before training and validation scoring, so the audit compares bullish/bearish directional performance only.

In [8]:
# CHANGED in clean notebook:
# The horizon audit now runs before the final PRIMARY_HORIZON model_df is constructed.
# It uses the horizon-independent feature_df and merges each candidate horizon's returns inside the loop.
# Training + validation only are used here; the held-out test period is not used for horizon choice.

# SELECTIVE CHANGE:
# Full classifier uses "full_ltn_qwen_plus_context".
# Selective notebook uses the matching bullish/bearish-only variant.
HORIZON_AUDIT_VARIANT = "selective_ltn_qwen_plus_context"


def build_model_df_for_horizon(horizon: int) -> pd.DataFrame:
    horizon_returns = simple_returns.loc[
        simple_returns["horizon_days"].eq(int(horizon)),
        ["ticker", "event_date", "simple_return", "prev_close", "future_close"],
    ]

    # Adding .copy() at the end of the merge chain consolidates 
    # the fragmented block manager into a clean, single memory block.
    out = (
        feature_df
        .drop(columns=["simple_return", "prev_close", "future_close", "target_label"], errors="ignore")
        .merge(
            horizon_returns,
            on=["ticker", "event_date"],
            how="left",
        )
    ).copy() 

    missing_returns = int(out["simple_return"].isna().sum())
    if missing_returns:
        print(f"{horizon}d horizon: dropping {missing_returns} rows with missing future returns")
        out = out.loc[out["simple_return"].notna()].copy()

    out["target_label"] = out["simple_return"].map(
        lambda x: realised_label_from_return(float(x), RETURN_THRESHOLD)
    )

    # SELECTIVE CHANGE:
    # Full classifier keeps bearish/bullish/neutral rows.
    # Selective notebook removes neutral rows after labelling each horizon.
    out = out.loc[out["target_label"].isin(LABELS)].copy()

    return out


horizon_ltn_rows = []

for horizon in HORIZON_CANDIDATES:
    print(f"LTN horizon audit: {horizon}-day simple return")

    model_df_h = build_model_df_for_horizon(horizon)

    train_h = model_df_h.loc[model_df_h["period"].eq("training")].copy()
    validation_h = model_df_h.loc[model_df_h["period"].eq("validation")].copy()
    train_val_h = model_df_h.loc[
        model_df_h["period"].isin(["training", "validation"])
    ].copy()

    spec_base_h = variant_specs[HORIZON_AUDIT_VARIANT].copy()

    candidate_rules_h, selected_rules_h = mine_validated_rules(
        train_val_h,
        spec_base_h["feature_cols"],
        spec_base_h["return_col"],
        rule_blocks=spec_base_h.get("rule_blocks"),
        max_pair_features=spec_base_h.get("max_pair_features", 36),
        max_rules_per_label=spec_base_h.get("max_rules_per_label", 12),
    )

    spec_h = spec_base_h.copy()
    spec_h["selected_rules"] = selected_rules_h.copy()

    bundle_h = train_variant(
        f"{HORIZON_AUDIT_VARIANT}_horizon_{horizon}d",
        spec_h,
        train_h,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

    scored_validation_h = score_variant(bundle_h, validation_h)

    row_h = summary_row(
        f"LTN horizon {horizon}d",
        "validation",
        scored_validation_h,
        bundle=bundle_h,
    )

    majority_baseline_h = validation_h["target_label"].value_counts(normalize=True).max() * 100

    row_h["horizon_days"] = horizon
    row_h["training_n"] = len(train_h)
    row_h["validation_n"] = len(validation_h)
    row_h["validation_majority_baseline_pct"] = float(majority_baseline_h)
    row_h["validation_bullish_n"] = int(validation_h["target_label"].eq("bullish").sum())
    row_h["validation_bearish_n"] = int(validation_h["target_label"].eq("bearish").sum())

    # SELECTIVE CHANGE:
    # Full classifier includes validation_neutral_n.
    # Selective notebook has already removed neutral rows, so that column is omitted.
    row_h["n_candidate_rules"] = len(candidate_rules_h)
    row_h["n_selected_rules"] = len(selected_rules_h)

    horizon_ltn_rows.append(row_h)


horizon_ltn_audit = pd.DataFrame(horizon_ltn_rows)

display(
    horizon_ltn_audit.sort_values(
        ["accuracy_pct", "aligned_mean_pct", "n_selected_rules"],
        ascending=False,
    )
)

LTN horizon audit: 1-day simple return
LTN horizon audit: 2-day simple return
LTN horizon audit: 3-day simple return
3d horizon: dropping 1 rows with missing future returns
LTN horizon audit: 4-day simple return
4d horizon: dropping 1 rows with missing future returns
LTN horizon audit: 5-day simple return
5d horizon: dropping 2 rows with missing future returns


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules,horizon_days,training_n,validation_n,validation_majority_baseline_pct,validation_bullish_n,validation_bearish_n,n_candidate_rules
3,LTN horizon 4d,validation,126,51.587302,0.381129,0.421874,0.394698,55.555556,44.444444,0.746239,True,0.3,21,4,161,126,54.761905,69,57,738
1,LTN horizon 2d,validation,118,50.847458,0.239925,-0.104632,0.463352,56.779661,43.220339,0.782354,True,0.3,15,2,147,118,55.084746,65,53,742
2,LTN horizon 3d,validation,120,50.000000,0.599657,-0.001903,0.536342,50.833333,49.166667,0.774638,True,0.3,17,3,152,120,57.500000,69,51,724
0,LTN horizon 1d,validation,115,47.826087,0.289431,-0.049310,0.711994,46.086957,53.913043,0.790763,True,0.3,18,1,138,115,58.260870,67,48,690
4,LTN horizon 5d,validation,123,42.276423,0.275722,-0.866000,0.964549,69.105691,30.894309,0.848206,True,0.3,22,5,154,123,51.219512,60,63,740


### 8. Select Primary Horizon And Build Final Model Frame

After the horizon audit, the selected `PRIMARY_HORIZON` is attached to the feature frame to create the final `model_df`. This is the frame used for frozen-rule refitting and held-out testing.

For the selective notebook, `model_df` contains only bullish and bearish realised-return rows. Neutral rows are labelled during horizon construction, then removed before final modelling.

In [9]:
# based on validation accuracy, update returns horizon to 4 days.
PRIMARY_HORIZON = 4

In [10]:
# CHANGED in clean notebook:
# Construct the final modelling frame only after the validation-only horizon audit.
# PRIMARY_HORIZON remains the configured retained horizon for the final held-out evaluation.

model_df = build_model_df_for_horizon(PRIMARY_HORIZON)

# SELECTIVE CHANGE:
# No extra filtering is needed here if build_model_df_for_horizon() already
# removes neutral rows. This assertion makes that explicit and catches mistakes.
unexpected_labels = sorted(set(model_df["target_label"]) - set(LABELS))
if unexpected_labels:
    raise ValueError(f"Selective model_df contains unexpected labels: {unexpected_labels}")

train = model_df.loc[model_df["period"].eq("training")].copy()
validation = model_df.loc[model_df["period"].eq("validation")].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

display(model_df.groupby("period")["target_label"].value_counts().unstack(fill_value=0))

4d horizon: dropping 1 rows with missing future returns


target_label,bearish,bullish
period,,
test,33,25
training,76,85
validation,57,69


## 9. Mine/Select Rules, Freeze Them, Then Refit

Candidate rules are selected using training plus validation only. The selected rules are frozen before the final model is refit on training plus validation.

### LTN variants experiment 1

This experiment compares selective LTN feature variants after the primary horizon has been selected. Rules are selected using training and validation data, then frozen. Each variant is refit on training plus validation and evaluated once on the held-out test set.

For the selective notebook, all rows in this experiment are directional bullish/bearish rows only; neutral rows have already been removed from `model_df`.

In [11]:
# no structural changes
# This cell is the same frozen-rule/refit workflow as the full-classifier notebook.
# SELECTIVE CHANGE:
# The only differences are output filenames and the fact that model_df already
# contains only bullish/bearish rows from the selective build_model_df_for_horizon().

train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()

frozen_variant_specs = {}

for name, spec in variant_specs.items():
    print("selecting frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_variant_specs[name] = frozen_spec


trained_final = {}

for name, spec in frozen_variant_specs.items():
    print("final frozen-rule training", name, "rows", len(train_val))

    trained_final[name] = train_variant(
        name,
        spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )


final_rows = []
final_frames = []

for variant, bundle in trained_final.items():
    scored = score_variant(bundle, test)

    final_rows.append(
        summary_row(
            variant,
            "test_frozen_rules_refit_training_validation",
            scored,
            bundle=bundle,
        )
    )

    final_frames.append(
        scored.assign(
            variant=variant,
            period="test_frozen_rules_refit_training_validation",
        )
    )


final_summary = pd.DataFrame(final_rows)

if SAVE_OUTPUTS:
    final_summary.to_csv(
        OUTPUT_DIR / "selective_ltn_final_frozen_rule_test_summary.csv",
        index=False,
    )

    pd.concat(final_frames, ignore_index=True).to_csv(
        OUTPUT_DIR / "selective_ltn_final_frozen_rule_test_predictions.csv",
        index=False,
    )


display(
    final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False,
    )
)

selecting frozen rules selective_ltn_qwen_only
selecting frozen rules selective_ltn_context_only
selecting frozen rules selective_ltn_qwen_plus_context
selecting frozen rules selective_ltn_qwen_context_interaction
final frozen-rule training selective_ltn_qwen_only rows 287
final frozen-rule training selective_ltn_context_only rows 287
final frozen-rule training selective_ltn_qwen_plus_context rows 287
final frozen-rule training selective_ltn_qwen_context_interaction rows 287


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
2,selective_ltn_qwen_plus_context,test_frozen_rules_refit_training_validation,58,60.344828,0.372654,0.818697,0.074003,48.275862,51.724138,0.785964,True,0.3,21
0,selective_ltn_qwen_only,test_frozen_rules_refit_training_validation,58,60.344828,0.372654,0.806717,0.074003,55.172414,44.827586,0.640796,True,0.3,16
3,selective_ltn_qwen_context_interaction,test_frozen_rules_refit_training_validation,58,58.620690,0.372654,0.527941,0.118524,50.000000,50.000000,0.865151,True,0.3,24
1,selective_ltn_context_only,test_frozen_rules_refit_training_validation,58,44.827586,0.372654,-0.496297,0.820928,46.551724,53.448276,0.676305,True,0.3,6


## 10. LTN variants experiment 2 - test on longer window

In [12]:
# no structural changes
# This is the same longer-window robustness workflow as the full-classifier notebook.
# SELECTIVE CHANGE:
# model_df already contains only bullish/bearish rows, so the robustness split
# remains selective without needing an additional neutral filter.

# Robustness check: longer held-out test window
# Training: existing early period
# Validation: first half of 2025
# Test: 2025-07-01 to 2026-06-01
#
# This preserves temporal ordering:
# training -> validation -> test

ROBUST_TRAINING = ("2023-06-01", "2024-12-31")
ROBUST_VALIDATION = ("2025-01-01", "2025-06-30")
ROBUST_TEST = ("2025-07-01", "2026-06-01")

def assign_robust_periods(df, date_col="event_date"):
    out = df.copy()
    d = pd.to_datetime(out[date_col]).dt.tz_localize(None)

    out["period_robust"] = np.select(
        [
            d.between(pd.Timestamp(ROBUST_TRAINING[0]), pd.Timestamp(ROBUST_TRAINING[1]), inclusive="both"),
            d.between(pd.Timestamp(ROBUST_VALIDATION[0]), pd.Timestamp(ROBUST_VALIDATION[1]), inclusive="both"),
            d.between(pd.Timestamp(ROBUST_TEST[0]), pd.Timestamp(ROBUST_TEST[1]), inclusive="both"),
        ],
        ["training", "validation", "test"],
        default="outside",
    )
    return out

model_df_robust = assign_robust_periods(model_df, "event_date")

model_df_robust = model_df_robust.loc[
    model_df_robust["period_robust"].isin(["training", "validation", "test"])
].copy()

model_df_robust["period_original"] = model_df_robust["period"]
model_df_robust["period"] = model_df_robust["period_robust"]

print("Robustness split counts:")
display(model_df_robust.groupby("period")["target_label"].value_counts().unstack(fill_value=0))

train_val_robust = model_df_robust.loc[
    model_df_robust["period"].isin(["training", "validation"])
].copy()

test_robust = model_df_robust.loc[
    model_df_robust["period"].eq("test")
].copy()

frozen_robust_variant_specs = {}

for name, spec in variant_specs.items():
    print("selecting robust frozen rules", name)

    candidate_rules, selected_rules = mine_validated_rules(
        train_val_robust,
        spec["feature_cols"],
        spec["return_col"],
        rule_blocks=spec.get("rule_blocks"),
        max_pair_features=spec.get("max_pair_features", 36),
        max_rules_per_label=spec.get("max_rules_per_label", 12),
    )

    frozen_spec = spec.copy()
    frozen_spec["selected_rules"] = selected_rules.copy()
    frozen_robust_variant_specs[name] = frozen_spec


trained_robust = {}

for name, spec in frozen_robust_variant_specs.items():
    print("robust frozen-rule training", name, "rows", len(train_val_robust))

    trained_robust[name] = train_variant(
        name,
        spec,
        train_val_robust,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=SEED,
        mine_rules=False,
    )

robust_rows = []
robust_frames = []

for variant, bundle in trained_robust.items():
    scored = score_variant(bundle, test_robust)  # Score the longer-window held-out test set.
    
    robust_rows.append(
        summary_row(
            variant,
            "longer_heldout_test_2025_07_to_2026_06",
            scored,
            bundle=bundle,
        )
    )
    
    robust_frames.append(
        scored.assign(
            variant=variant,
            period="longer_heldout_test_2025_07_to_2026_06",
        )
    )

robust_final_summary = pd.DataFrame(robust_rows)
robust_final_predictions = pd.concat(robust_frames, ignore_index=True)

display(
    robust_final_summary.sort_values(
        ["accuracy_pct", "aligned_mean_pct"],
        ascending=False
    )
)

if SAVE_OUTPUTS:
    robust_final_summary.to_csv(
        OUTPUT_DIR / "selective_ltn_robust_longer_test_summary.csv",
        index=False,
    )

    robust_final_predictions.to_csv(
        OUTPUT_DIR / "selective_ltn_robust_longer_test_predictions.csv",
        index=False,
    )

Robustness split counts:


target_label,bearish,bullish
period,,
test,58,65
training,76,85
validation,32,29


selecting robust frozen rules selective_ltn_qwen_only
selecting robust frozen rules selective_ltn_context_only
selecting robust frozen rules selective_ltn_qwen_plus_context
selecting robust frozen rules selective_ltn_qwen_context_interaction
robust frozen-rule training selective_ltn_qwen_only rows 222
robust frozen-rule training selective_ltn_context_only rows 222
robust frozen-rule training selective_ltn_qwen_plus_context rows 222
robust frozen-rule training selective_ltn_qwen_context_interaction rows 222


,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,selective_ltn_qwen_only,longer_heldout_test_2025_07_to_2026_06,123,58.536585,0.879553,0.672056,0.035451,56.910569,43.089431,0.636199,True,0.3,10
2,selective_ltn_qwen_plus_context,longer_heldout_test_2025_07_to_2026_06,123,57.723577,0.879553,0.358324,0.052102,56.097561,43.902439,0.789504,True,0.3,20
1,selective_ltn_context_only,longer_heldout_test_2025_07_to_2026_06,123,56.097561,0.879553,-0.129755,0.103324,43.089431,56.910569,0.770257,True,0.3,2
3,selective_ltn_qwen_context_interaction,longer_heldout_test_2025_07_to_2026_06,123,54.471545,0.879553,0.056614,0.183648,51.219512,48.780488,0.749506,True,0.3,24


## 11. LTN vs no LTN ablation test 1

This section compares the selective LTN Qwen-plus-context model against a neural classifier using the same feature set with logic disabled.

For the selective notebook, both branches use only bullish/bearish directional rows. Neutral rows are not included in either the LTN branch or the no-logic neural branch.

In [13]:
# no structural changes
# Same LTN-vs-no-logic ablation structure as the full-classifier notebook.
# SELECTIVE CHANGE:
# Use the selective Qwen-plus-context variant and selective output names.
# model_df already contains only bullish/bearish rows.

train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

ablation_base_spec = variant_specs["selective_ltn_qwen_plus_context"]


candidate_rules, selected_rules = mine_validated_rules(
    train_val,
    ablation_base_spec["feature_cols"],
    ablation_base_spec["return_col"],
    rule_blocks=ablation_base_spec.get("rule_blocks"),
    max_pair_features=ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=ablation_base_spec.get("max_rules_per_label", 12),
)

ablation_ltn_spec = ablation_base_spec.copy()
ablation_ltn_spec["selected_rules"] = selected_rules.copy()

# display(pd.DataFrame([{
#     "variant": "selective_ltn_qwen_plus_context",
#     "rule_blocks": ", ".join(ablation_base_spec.get("rule_blocks", [])),
#     "n_candidate_rules": len(candidate_rules),
#     "n_selected_rules": len(selected_rules),
#     "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
#     "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
#     "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
#     "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
# }]))


ltn_bundle = train_variant(
    "selective_ltn_qwen_plus_context_ablation",
    ablation_ltn_spec,
    train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

ltn_scored = score_variant(ltn_bundle, test)


no_logic_bundle = train_variant(
    "selective_no_logic_qwen_plus_context_ablation",
    ablation_base_spec,
    train_val,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

no_logic_scored = score_variant(no_logic_bundle, test)


ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "Selective LTN, Qwen + context",
        "heldout_test",
        ltn_scored,
        bundle=ltn_bundle,
    ),
    summary_row(
        "Neural directional classifier, Qwen + context, no logic",
        "heldout_test",
        no_logic_scored,
        bundle=no_logic_bundle,
    ),
])

display(ltn_ablation_summary)


if SAVE_OUTPUTS:
    ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "selective_ltn_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "selective_no_logic_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    ltn_scored.to_csv(
        OUTPUT_DIR / "selective_ltn_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    no_logic_scored.to_csv(
        OUTPUT_DIR / "selective_no_logic_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "selective_ltn_ablation_qwen_plus_context_summary.csv",
        index=False,
    )

,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,"Selective LTN, Qwen + context",heldout_test,58,58.620690,0.372654,0.731391,0.118524,50.000000,50.000000,0.828239,True,0.3,21
1,"Neural directional classifier, Qwen + context,...",heldout_test,58,55.172414,0.372654,0.319590,0.255921,46.551724,53.448276,0.898517,False,0.0,0


## 12. LTN vs no LTN ablation test 2

same ablation experiment as test one, but over repeated seeds


In [14]:
# no structural changes
# Same repeated-seed ablation structure as the full-classifier notebook.
# SELECTIVE CHANGE:
# Use the selective Qwen-plus-context variant and selective model names.
# model_df already contains only bullish/bearish rows.

seed_rows = []
seed_rule_rows = []

train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
test = model_df.loc[model_df["period"].eq("test")].copy()

base_spec = variant_specs["selective_ltn_qwen_plus_context"]

for seed in ROBUST_SEEDS:
    print(f"Main repeated-seed ablation: seed {seed}")

    candidate_rules, selected_rules = mine_validated_rules(
        train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    ltn_spec = base_spec.copy()
    ltn_spec["selected_rules"] = selected_rules.copy()

    seed_rule_rows.append({
        "seed": seed,
        "model": "Selective LTN, Qwen + context",
        "rule_blocks": ", ".join(base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    # CHANGED in clean notebook:
    # LTN branch uses frozen rules, so mine_rules=False.
    ltn_bundle = train_variant(
        f"selective_ltn_qwen_plus_context_seed_{seed}",
        ltn_spec,
        train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    ltn_scored = score_variant(ltn_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "Selective LTN, Qwen + context",
        **summary_row("Selective LTN, Qwen + context", "heldout_test", ltn_scored, bundle=ltn_bundle),
    })

    no_logic_bundle = train_variant(
        f"selective_no_logic_qwen_plus_context_seed_{seed}",
        base_spec,
        train_val,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    no_logic_scored = score_variant(no_logic_bundle, test)

    seed_rows.append({
        "seed": seed,
        "model": "Neural directional classifier, Qwen + context, no logic",
        **summary_row(
            "Neural directional classifier, Qwen + context, no logic",
            "heldout_test",
            no_logic_scored,
            bundle=no_logic_bundle,
        ),
    })


seed_results = pd.DataFrame(seed_rows)
seed_rule_summary = pd.DataFrame(seed_rule_rows)

seed_summary = (
    seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

display(seed_rule_summary)
display(seed_results)
display(seed_summary)

Main repeated-seed ablation: seed 1
Main repeated-seed ablation: seed 2
Main repeated-seed ablation: seed 3
Main repeated-seed ablation: seed 4
Main repeated-seed ablation: seed 5
Main repeated-seed ablation: seed 7
Main repeated-seed ablation: seed 11
Main repeated-seed ablation: seed 13
Main repeated-seed ablation: seed 17
Main repeated-seed ablation: seed 19


,seed,model,rule_blocks,n_candidate_rules,n_selected_rules,validation_n_min,validation_n_median,validation_n_max,validation_accuracy_mean
0,1,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
1,2,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
2,3,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
3,4,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
4,5,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
5,7,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
6,11,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
7,13,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
8,17,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314
9,19,"Selective LTN, Qwen + context","qwen_semantic, market_context",738,21,5,8.0,23,75.890314


,seed,model,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,1,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",heldout_test,58,56.896552,0.372654,0.229049,0.179072,48.275862,51.724138,0.839759,True,0.3,21
1,1,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",heldout_test,58,56.896552,0.372654,0.197290,0.179072,48.275862,51.724138,0.813400,False,0.0,0
2,2,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",heldout_test,58,56.896552,0.372654,0.243133,0.179072,51.724138,48.275862,0.747415,True,0.3,21
3,2,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",heldout_test,58,53.448276,0.372654,0.089058,0.347002,48.275862,51.724138,0.786888,False,0.0,0
4,3,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",heldout_test,58,51.724138,0.372654,-0.136152,0.447842,39.655172,60.344828,0.886353,True,0.3,21
5,3,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",heldout_test,58,55.172414,0.372654,0.222141,0.255921,50.000000,50.000000,0.845642,False,0.0,0
6,4,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",heldout_test,58,60.344828,0.372654,0.323479,0.074003,44.827586,55.172414,0.848798,True,0.3,21
7,4,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",heldout_test,58,58.620690,0.372654,0.347961,0.118524,46.551724,53.448276,0.814913,False,0.0,0
8,5,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",heldout_test,58,60.344828,0.372654,0.742388,0.074003,55.172414,44.827586,0.863799,True,0.3,21
9,5,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",heldout_test,58,55.172414,0.372654,0.337133,0.255921,50.000000,50.000000,0.809420,False,0.0,0


,model,runs,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"Neural directional classifier, Qwen + context,...",10,55.344828,2.064171,0.215442,0.133398,0.818899,0.032643,0.0,0,0
1,"Selective LTN, Qwen + context",10,57.758621,2.958514,0.403987,0.313204,0.823296,0.047102,21.0,21,21


## 13. LTN vs no-LTN test 3 - Longer-Window Robustness

This section repeats the frozen-rule workflow under a longer held-out test window. It keeps temporal ordering as training, validation, then test.

For the selective notebook, the longer-window comparison is still restricted to bullish/bearish directional rows. Neutral rows are not included in either the LTN branch or the no-logic branch.

In [15]:
# no structural changes
# Same longer-window LTN-vs-no-logic ablation structure as the full-classifier notebook.
# SELECTIVE CHANGE:
# Use the selective Qwen-plus-context variant and selective model/output names.
# train_val_robust and test_robust already contain only bullish/bearish rows.

robust_ablation_base_spec = variant_specs["selective_ltn_qwen_plus_context"]


robust_candidate_rules, robust_selected_rules = mine_validated_rules(
    train_val_robust,
    robust_ablation_base_spec["feature_cols"],
    robust_ablation_base_spec["return_col"],
    rule_blocks=robust_ablation_base_spec.get("rule_blocks"),
    max_pair_features=robust_ablation_base_spec.get("max_pair_features", 36),
    max_rules_per_label=robust_ablation_base_spec.get("max_rules_per_label", 12),
)

robust_ablation_ltn_spec = robust_ablation_base_spec.copy()
robust_ablation_ltn_spec["selected_rules"] = robust_selected_rules.copy()

# display(pd.DataFrame([{
#     "variant": "selective_ltn_qwen_plus_context",
#     "rule_blocks": ", ".join(robust_ablation_base_spec.get("rule_blocks", [])),
#     "n_candidate_rules": len(robust_candidate_rules),
#     "n_selected_rules": len(robust_selected_rules),
#     "validation_n_min": robust_selected_rules["validation_n"].min() if len(robust_selected_rules) else np.nan,
#     "validation_n_median": robust_selected_rules["validation_n"].median() if len(robust_selected_rules) else np.nan,
#     "validation_n_max": robust_selected_rules["validation_n"].max() if len(robust_selected_rules) else np.nan,
#     "validation_accuracy_mean": robust_selected_rules["validation_accuracy_pct"].mean() if len(robust_selected_rules) else np.nan,
# }]))


robust_ltn_bundle = train_variant(
    "selective_robust_ltn_qwen_plus_context_ablation",
    robust_ablation_ltn_spec,
    train_val_robust,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

robust_ltn_scored = score_variant(
    robust_ltn_bundle,
    test_robust,
)


robust_no_logic_bundle = train_variant(
    "selective_robust_no_logic_qwen_plus_context_ablation",
    robust_ablation_base_spec,
    train_val_robust,
    use_logic=False,
    logic_weight=0.0,
    seed=SEED,
    mine_rules=False,
)

robust_no_logic_scored = score_variant(
    robust_no_logic_bundle,
    test_robust,
)


robust_ltn_ablation_summary = pd.DataFrame([
    summary_row(
        "Selective LTN, Qwen + context",
        "longer_heldout_test_2025_07_to_2026_06",
        robust_ltn_scored,
        bundle=robust_ltn_bundle,
    ),
    summary_row(
        "Neural directional classifier, Qwen + context, no logic",
        "longer_heldout_test_2025_07_to_2026_06",
        robust_no_logic_scored,
        bundle=robust_no_logic_bundle,
    ),
])

display(robust_ltn_ablation_summary)


if SAVE_OUTPUTS:
    robust_ltn_bundle["history"].to_csv(
        OUTPUT_DIR / "selective_robust_ltn_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    robust_no_logic_bundle["history"].to_csv(
        OUTPUT_DIR / "selective_robust_no_logic_qwen_plus_context_ablation_training_history.csv",
        index=False,
    )

    robust_ltn_scored.to_csv(
        OUTPUT_DIR / "selective_robust_ltn_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    robust_no_logic_scored.to_csv(
        OUTPUT_DIR / "selective_robust_no_logic_qwen_plus_context_ablation_test_predictions.csv",
        index=False,
    )

    robust_ltn_ablation_summary.to_csv(
        OUTPUT_DIR / "selective_robust_ltn_ablation_qwen_plus_context_summary.csv",
        index=False,
    )

,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,"Selective LTN, Qwen + context",longer_heldout_test_2025_07_to_2026_06,123,56.097561,0.879553,0.277525,0.103324,57.723577,42.276423,0.812219,True,0.3,20
1,"Neural directional classifier, Qwen + context,...",longer_heldout_test_2025_07_to_2026_06,123,60.162602,0.879553,0.434742,0.015023,52.032520,47.967480,0.768276,False,0.0,0


## 14. LTN vs No LTN test 4 - longer window robust set and repeated seeds

This section repeats the main and longer-window LTN/no-logic comparisons across multiple random seeds.

For the selective notebook, the repeated-seed comparison remains restricted to bullish/bearish directional rows. Neutral rows are not included in the seed-level summaries or confusion matrices.

In [16]:
# no structural changes
# Same longer-window repeated-seed workflow as the full-classifier notebook.
# SELECTIVE CHANGE:
# Use the selective Qwen-plus-context variant and remove neutral-rate aggregation.
# Confusion matrices still use LABELS, but LABELS = ["bearish", "bullish"] here.

robust_seed_rows = []
robust_seed_rule_rows = []
robust_seed_scored_frames = []

robust_base_spec = variant_specs["selective_ltn_qwen_plus_context"]

for seed in ROBUST_SEEDS:
    print(f"Longer-window repeated seed: {seed}")

    robust_candidate_rules, robust_selected_rules = mine_validated_rules(
        train_val_robust,
        robust_base_spec["feature_cols"],
        robust_base_spec["return_col"],
        rule_blocks=robust_base_spec.get("rule_blocks"),
        max_pair_features=robust_base_spec.get("max_pair_features", 36),
        max_rules_per_label=robust_base_spec.get("max_rules_per_label", 12),
    )

    robust_seed_rule_rows.append({
        "seed": seed,
        "rule_blocks": ", ".join(robust_base_spec.get("rule_blocks", [])),
        "n_candidate_rules": len(robust_candidate_rules),
        "n_selected_rules": len(robust_selected_rules),
    })

    robust_ltn_spec = robust_base_spec.copy()
    robust_ltn_spec["selected_rules"] = robust_selected_rules.copy()

    robust_ltn_bundle = train_variant(
        f"selective_robust_ltn_qwen_plus_context_seed_{seed}",
        robust_ltn_spec,
        train_val_robust,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    robust_ltn_scored = score_variant(robust_ltn_bundle, test_robust)

    robust_seed_scored_frames.append(
        robust_ltn_scored.assign(
            seed=seed,
            model="Selective LTN, Qwen + context",
        )
    )

    robust_seed_rows.append({
        "seed": seed,
        "model": "Selective LTN, Qwen + context",
        **summary_row(
            "Selective LTN, Qwen + context",
            "longer_heldout_test",
            robust_ltn_scored,
            bundle=robust_ltn_bundle,
        ),
    })

    robust_no_logic_bundle = train_variant(
        f"selective_robust_no_logic_qwen_plus_context_seed_{seed}",
        robust_base_spec,
        train_val_robust,
        use_logic=False,
        logic_weight=0.0,
        seed=seed,
        mine_rules=False,
    )

    robust_no_logic_scored = score_variant(robust_no_logic_bundle, test_robust)

    robust_seed_scored_frames.append(
        robust_no_logic_scored.assign(
            seed=seed,
            model="Neural directional classifier, Qwen + context, no logic",
        )
    )

    robust_seed_rows.append({
        "seed": seed,
        "model": "Neural directional classifier, Qwen + context, no logic",
        **summary_row(
            "Neural directional classifier, Qwen + context, no logic",
            "longer_heldout_test",
            robust_no_logic_scored,
            bundle=robust_no_logic_bundle,
        ),
    })

robust_seed_results = pd.DataFrame(robust_seed_rows)
robust_seed_rule_summary = pd.DataFrame(robust_seed_rule_rows)

robust_seed_predictions = pd.concat(robust_seed_scored_frames, ignore_index=True)

robust_seed_summary = (
    robust_seed_results
    .groupby("model")
    .agg(
        runs=("seed", "nunique"),
        events_mean=("n", "mean"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        aligned_mean_return_mean=("aligned_mean_pct", "mean"),
        aligned_mean_return_std=("aligned_mean_pct", "std"),
        signflip_p_mean=("aligned_signflip_p", "mean"),
        bullish_rate_mean=("bullish_pred_rate_pct", "mean"),
        bearish_rate_mean=("bearish_pred_rate_pct", "mean"),

        # SELECTIVE CHANGE:
        # Full classifier also aggregates neutral_rate_mean.
        # Selective notebook has no neutral_pred_rate_pct column.
        confidence_margin_mean=("mean_confidence_margin", "mean"),
        confidence_margin_std=("mean_confidence_margin", "std"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
    .reset_index()
)

robust_confusion_tables = {}

for model_name, group in robust_seed_predictions.groupby("model"):
    cm_counts = pd.crosstab(
        group["target_label"],
        group["prediction"],
        rownames=["True label"],
        colnames=["Predicted label"],
        dropna=False,
    ).reindex(index=LABELS, columns=LABELS, fill_value=0)

    cm_row_pct = (
        cm_counts
        .div(cm_counts.sum(axis=1).replace(0, np.nan), axis=0)
        .mul(100)
        .round(1)
    )

    robust_confusion_tables[model_name] = {
        "counts": cm_counts,
        "row_pct": cm_row_pct,
    }

display(robust_seed_rule_summary)
display(robust_seed_results)
display(robust_seed_summary)

for model_name, tables in robust_confusion_tables.items():
    print(model_name)
    display(tables["counts"])
    display(tables["row_pct"])

Longer-window repeated seed: 1
Longer-window repeated seed: 2
Longer-window repeated seed: 3
Longer-window repeated seed: 4
Longer-window repeated seed: 5
Longer-window repeated seed: 7
Longer-window repeated seed: 11
Longer-window repeated seed: 13
Longer-window repeated seed: 17
Longer-window repeated seed: 19


,seed,rule_blocks,n_candidate_rules,n_selected_rules
0,1,"qwen_semantic, market_context",556,20
1,2,"qwen_semantic, market_context",556,20
2,3,"qwen_semantic, market_context",556,20
3,4,"qwen_semantic, market_context",556,20
4,5,"qwen_semantic, market_context",556,20
5,7,"qwen_semantic, market_context",556,20
6,11,"qwen_semantic, market_context",556,20
7,13,"qwen_semantic, market_context",556,20
8,17,"qwen_semantic, market_context",556,20
9,19,"qwen_semantic, market_context",556,20


,seed,model,variant,period,n,accuracy_pct,mean_simple_return_pct,aligned_mean_pct,aligned_signflip_p,bullish_pred_rate_pct,bearish_pred_rate_pct,mean_confidence_margin,use_logic,logic_weight,n_selected_rules
0,1,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",longer_heldout_test,123,56.910569,0.879553,0.189990,0.074408,53.658537,46.341463,0.736886,True,0.3,20
1,1,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",longer_heldout_test,123,54.471545,0.879553,0.096721,0.183648,49.593496,50.406504,0.744745,False,0.0,0
2,2,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",longer_heldout_test,123,56.910569,0.879553,0.223412,0.074408,56.910569,43.089431,0.788005,True,0.3,20
3,2,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",longer_heldout_test,123,58.536585,0.879553,0.336959,0.035451,58.536585,41.463415,0.773811,False,0.0,0
4,3,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",longer_heldout_test,123,56.097561,0.879553,-0.214220,0.103324,46.341463,53.658537,0.722839,True,0.3,20
5,3,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",longer_heldout_test,123,55.284553,0.879553,0.316866,0.139599,55.284553,44.715447,0.710817,False,0.0,0
6,4,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",longer_heldout_test,123,56.910569,0.879553,0.345592,0.074408,52.032520,47.967480,0.816534,True,0.3,20
7,4,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",longer_heldout_test,123,55.284553,0.879553,0.443683,0.139599,56.910569,43.089431,0.843831,False,0.0,0
8,5,"Selective LTN, Qwen + context","Selective LTN, Qwen + context",longer_heldout_test,123,56.910569,0.879553,0.317870,0.074408,55.284553,44.715447,0.813500,True,0.3,20
9,5,"Neural directional classifier, Qwen + context,...","Neural directional classifier, Qwen + context,...",longer_heldout_test,123,55.284553,0.879553,-0.007939,0.139599,53.658537,46.341463,0.771361,False,0.0,0


,model,runs,events_mean,accuracy_mean,accuracy_std,aligned_mean_return_mean,aligned_mean_return_std,signflip_p_mean,bullish_rate_mean,bearish_rate_mean,confidence_margin_mean,confidence_margin_std,selected_rules_mean,selected_rules_min,selected_rules_max
0,"Neural directional classifier, Qwen + context,...",10,123.0,56.829268,2.007982,0.295394,0.188477,0.094245,53.739837,46.260163,0.772459,0.050991,0.0,0,0
1,"Selective LTN, Qwen + context",10,123.0,57.235772,1.679342,0.249644,0.193898,0.073975,54.634146,45.365854,0.773682,0.042570,20.0,20,20


Neural directional classifier, Qwen + context, no logic


Predicted label,bearish,bullish
True label,,
bearish,309,271
bullish,260,390


Predicted label,bearish,bullish
True label,,
bearish,53.3,46.7
bullish,40.0,60.0


Selective LTN, Qwen + context


Predicted label,bearish,bullish
True label,,
bearish,306,274
bullish,252,398


Predicted label,bearish,bullish
True label,,
bearish,52.8,47.2
bullish,38.8,61.2


## 15. Selective High-Confidence Threshold Filter

This section adds the selective-filter layer from the older MAG7 notebook onto the clean binary directional notebook. The model still scores bullish and bearish outcomes, but predictions are only emitted when the relevant score passes a validation-chosen confidence threshold. Rows below threshold are labelled `no_signal`.

The filter is split into two stages to avoid test leakage.

**15.1 Validation threshold quantile sweep**

Candidate rules are mined on the training period and filtered using the validation period. The selected rules are then frozen. A threshold-calibration model is trained on the training period only, scored on validation only, and used to inspect the accuracy/coverage trade-off across threshold quantiles. The held-out test set is not scored or inspected in this stage.

**15.2 Frozen-threshold held-out test**

After bullish and bearish thresholds have been selected using validation only, the thresholds are frozen. The final model is then refit on training plus validation using the already frozen rules, and the frozen thresholds are applied once to the held-out test set. Test accuracy and coverage are reported only at this final stage.les.

In [17]:
# Selective high-confidence threshold helpers.
#
# This cell defines the thresholding machinery used in Section 15.
# It does not score the held-out test set.

SELECTIVE_FILTER_VARIANT = "selective_ltn_qwen_plus_context"
THRESHOLD_QUANTILES = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85]

if SELECTIVE_FILTER_VARIANT not in frozen_variant_specs:
    raise KeyError(
        f"{SELECTIVE_FILTER_VARIANT!r} was not found in frozen_variant_specs. "
        f"Available variants: {list(frozen_variant_specs)}"
    )


def apply_selective_thresholds(scored, bullish_threshold, bearish_threshold):
    out = scored.copy()

    bullish_signal = (
        out["score_bullish"].ge(bullish_threshold)
        & out["score_bullish"].gt(out["score_bearish"])
    )
    bearish_signal = (
        out["score_bearish"].ge(bearish_threshold)
        & out["score_bearish"].gt(out["score_bullish"])
    )

    out["selective_prediction"] = "no_signal"
    out.loc[bullish_signal, "selective_prediction"] = "bullish"
    out.loc[bearish_signal, "selective_prediction"] = "bearish"
    out["has_signal"] = out["selective_prediction"].isin(LABELS)

    return out


def selective_subset_stats(df, variant, period, signal_label=None, threshold_quantile=None):
    if signal_label is None:
        subset = df.loc[df["has_signal"]].copy()
        label_name = "joint"
    else:
        subset = df.loc[df["selective_prediction"].eq(signal_label)].copy()
        label_name = signal_label

    row = {
        "variant": variant,
        "period": period,
        "signal": label_name,
        "threshold_quantile": threshold_quantile,
        "n_total": int(len(df)),
        "n_signal": int(len(subset)),
        "coverage_pct": float(100 * len(subset) / len(df)) if len(df) else np.nan,
    }

    if subset.empty:
        row.update({
            "accuracy_pct": np.nan,
            "aligned_mean_pct": np.nan,
            "aligned_signflip_p": np.nan,
            "mean_simple_return_pct": np.nan,
        })
        return row

    ret = pd.to_numeric(subset["simple_return"], errors="coerce")
    pred = subset["selective_prediction"]

    signed = np.select(
        [pred.eq("bullish"), pred.eq("bearish")],
        [ret, -ret],
        default=np.nan,
    )
    signed = pd.Series(signed, index=subset.index).dropna()

    if len(signed) >= 3:
        p = binomtest(
            int((signed > 0).sum()),
            len(signed),
            0.5,
            alternative="greater",
        ).pvalue
        aligned_mean = float(100 * signed.mean())
    else:
        p = np.nan
        aligned_mean = np.nan

    row.update({
        "accuracy_pct": float(100 * subset["selective_prediction"].eq(subset["target_label"]).mean()),
        "aligned_mean_pct": aligned_mean,
        "aligned_signflip_p": float(p) if np.isfinite(p) else np.nan,
        "mean_simple_return_pct": float(100 * ret.mean()),
    })

    return row


def choose_label_threshold(scored_validation, label, quantiles=THRESHOLD_QUANTILES):
    score_col = f"score_{label}"
    other_col = "score_bullish" if label == "bearish" else "score_bearish"

    rows = []

    for q in quantiles:
        threshold = float(scored_validation[score_col].quantile(q))

        tmp = scored_validation.copy()
        tmp["selective_prediction"] = "no_signal"

        mask = tmp[score_col].ge(threshold) & tmp[score_col].gt(tmp[other_col])
        tmp.loc[mask, "selective_prediction"] = label
        tmp["has_signal"] = tmp["selective_prediction"].eq(label)

        stats = selective_subset_stats(
            tmp,
            variant="threshold_search",
            period="validation",
            signal_label=label,
            threshold_quantile=q,
        )
        stats.update({
            "label": label,
            "quantile": q,
            "threshold": threshold,
        })

        rows.append(stats)

    search = pd.DataFrame(rows)
    eligible = search.loc[search["n_signal"].ge(MIN_VALIDATION_EVENTS)].copy()

    if eligible.empty:
        eligible = search.copy()

    chosen = (
        eligible
        .sort_values(
            ["accuracy_pct", "aligned_mean_pct", "n_signal", "quantile"],
            ascending=[False, False, False, False],
        )
        .iloc[0]
    )

    return chosen, search

### 15.1 Validation Threshold Quantile Sweep

This stage uses the training period to fit the threshold-calibration model and the validation period to select bullish and bearish confidence thresholds. The held-out test set is not scored or inspected here.

The table reports validation-only accuracy and coverage across candidate threshold quantiles, so the accuracy/coverage trade-off can be inspected before thresholds are frozen.

In [18]:
threshold_train = model_df.loc[model_df["period"].eq("training")].copy()
threshold_validation = model_df.loc[model_df["period"].eq("validation")].copy()

threshold_bundles = {}
threshold_rows = []
threshold_search_frames = []
threshold_grid_rows = []

name = SELECTIVE_FILTER_VARIANT
spec = frozen_variant_specs[name]

print("selective threshold calibration", name, "training rows", len(threshold_train))

threshold_bundles[name] = train_variant(
    f"{name}_threshold_calibration",
    spec,
    threshold_train,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

validation_scored = score_variant(threshold_bundles[name], threshold_validation)

bullish_choice, bullish_search = choose_label_threshold(validation_scored, "bullish")
bearish_choice, bearish_search = choose_label_threshold(validation_scored, "bearish")

bullish_threshold = float(bullish_choice["threshold"])
bearish_threshold = float(bearish_choice["threshold"])

threshold_rows.append({
    "variant": name,
    "bullish_quantile": float(bullish_choice["quantile"]),
    "bullish_threshold": bullish_threshold,
    "bullish_validation_n": int(bullish_choice["n_signal"]),
    "bullish_validation_accuracy_pct": float(bullish_choice["accuracy_pct"]),
    "bearish_quantile": float(bearish_choice["quantile"]),
    "bearish_threshold": bearish_threshold,
    "bearish_validation_n": int(bearish_choice["n_signal"]),
    "bearish_validation_accuracy_pct": float(bearish_choice["accuracy_pct"]),
    "n_selected_rules": len(threshold_bundles[name]["selected_rules"]),
})

threshold_search_frames.append(
    pd.concat([bullish_search, bearish_search], ignore_index=True).assign(variant=name)
)

for q in THRESHOLD_QUANTILES:
    bullish_q_threshold = float(validation_scored["score_bullish"].quantile(q))
    bearish_q_threshold = float(validation_scored["score_bearish"].quantile(q))

    validation_q = apply_selective_thresholds(
        validation_scored,
        bullish_q_threshold,
        bearish_q_threshold,
    )

    for signal_label in ["bullish", "bearish", None]:
        threshold_grid_rows.append(
            selective_subset_stats(
                validation_q,
                variant=name,
                period="validation_threshold_grid",
                signal_label=signal_label,
                threshold_quantile=q,
            )
        )

threshold_selection_summary = pd.DataFrame(threshold_rows)
threshold_search_results = pd.concat(threshold_search_frames, ignore_index=True)
threshold_grid_summary = pd.DataFrame(threshold_grid_rows)

# display(threshold_selection_summary)

display(
    threshold_grid_summary
    .sort_values(["threshold_quantile", "signal"])
    .loc[:, [
        "variant",
        "period",
        "signal",
        "threshold_quantile",
        "n_total",
        "n_signal",
        "coverage_pct",
        "accuracy_pct",
        "aligned_mean_pct",
        "aligned_signflip_p",
        "mean_simple_return_pct",
    ]]
)

selective threshold calibration selective_ltn_qwen_plus_context training rows 161


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct
1,selective_ltn_qwen_plus_context,validation_threshold_grid,bearish,0.50,126,50,39.682540,44.000000,0.106161,0.838882,-0.106161
0,selective_ltn_qwen_plus_context,validation_threshold_grid,bullish,0.50,126,58,46.031746,55.172414,1.081431,0.255921,1.081431
2,selective_ltn_qwen_plus_context,validation_threshold_grid,joint,0.50,126,108,85.714286,50.000000,0.629917,0.538299,0.531620
4,selective_ltn_qwen_plus_context,validation_threshold_grid,bearish,0.60,126,47,37.301587,40.425532,-0.112887,0.928067,0.112887
3,selective_ltn_qwen_plus_context,validation_threshold_grid,bullish,0.60,126,49,38.888889,55.102041,0.973002,0.284086,0.973002
5,selective_ltn_qwen_plus_context,validation_threshold_grid,joint,0.60,126,96,76.190476,47.916667,0.441369,0.694966,0.551904
7,selective_ltn_qwen_plus_context,validation_threshold_grid,bearish,0.70,126,35,27.777778,40.000000,0.003374,0.912267,-0.003374
6,selective_ltn_qwen_plus_context,validation_threshold_grid,bullish,0.70,126,38,30.158730,55.263158,0.493814,0.313551,0.493814
8,selective_ltn_qwen_plus_context,validation_threshold_grid,joint,0.70,126,73,57.936508,47.945205,0.258671,0.680014,0.255436
10,selective_ltn_qwen_plus_context,validation_threshold_grid,bearish,0.75,126,31,24.603175,38.709677,-0.549108,0.925194,0.549108


In [19]:
CONFIDENCE_THRESHOLD = 0.85

### 15.2 Frozen-Threshold Held-Out Test

This stage freezes the bullish and bearish thresholds selected from validation. The final model is then refit on training plus validation using the already frozen rules, and the frozen thresholds are applied once to the held-out test set.

The test set is used only in this final evaluation stage.

In [20]:
threshold_test = model_df.loc[model_df["period"].eq("test")].copy()
threshold_train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()

final_threshold_bundles = {}
selective_test_rows = []
selective_test_frames = []

name = SELECTIVE_FILTER_VARIANT
spec = frozen_variant_specs[name]

# Use the manually selected validation quantile.
bullish_threshold = float(validation_scored["score_bullish"].quantile(CONFIDENCE_THRESHOLD))
bearish_threshold = float(validation_scored["score_bearish"].quantile(CONFIDENCE_THRESHOLD))

manual_threshold_selection_summary = pd.DataFrame([{
    "variant": name,
    "confidence_threshold_quantile": CONFIDENCE_THRESHOLD,
    "bullish_threshold": bullish_threshold,
    "bearish_threshold": bearish_threshold,
    "selection_mode": "manual_validation_quantile",
}])

# display(manual_threshold_selection_summary)

print("final selective-filter refit", name, "training+validation rows", len(threshold_train_val))

final_threshold_bundles[name] = train_variant(
    f"{name}_final_threshold_refit",
    spec,
    threshold_train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

test_scored = score_variant(final_threshold_bundles[name], threshold_test)
test_selective = apply_selective_thresholds(
    test_scored,
    bullish_threshold,
    bearish_threshold,
)

for signal_label in ["bullish", "bearish", None]:
    selective_test_rows.append(
        selective_subset_stats(
            test_selective,
            variant=name,
            period="test_frozen_manual_validation_thresholds",
            signal_label=signal_label,
            threshold_quantile=CONFIDENCE_THRESHOLD,
        )
    )

selective_test_frames.append(
    test_selective.assign(
        variant=name,
        period="test_frozen_manual_validation_thresholds",
        confidence_threshold_quantile=CONFIDENCE_THRESHOLD,
        bullish_threshold=bullish_threshold,
        bearish_threshold=bearish_threshold,
    )
)

selective_high_confidence_summary = pd.DataFrame(selective_test_rows)
selective_high_confidence_predictions = pd.concat(selective_test_frames, ignore_index=True)

display(
    selective_high_confidence_summary.sort_values(
        ["signal", "accuracy_pct", "coverage_pct"],
        ascending=[True, False, False],
    )
)

final selective-filter refit selective_ltn_qwen_plus_context training+validation rows 287


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct
1,selective_ltn_qwen_plus_context,test_frozen_manual_validation_thresholds,bearish,0.85,58,20,34.482759,70.000000,0.929220,0.057659,-0.929220
0,selective_ltn_qwen_plus_context,test_frozen_manual_validation_thresholds,bullish,0.85,58,8,13.793103,62.500000,-0.047015,0.363281,-0.047015
2,selective_ltn_qwen_plus_context,test_frozen_manual_validation_thresholds,joint,0.85,58,28,48.275862,67.857143,0.650296,0.043579,-0.677161


### 15.3 Repeated-Seed Robustness Of The Selective Filter

This robustness check repeats the same selective-filter procedure across random seeds for the fixed `selective_ltn_qwen_plus_context` variant. The confidence threshold quantile is not re-selected per seed; each run uses the manually chosen `CONFIDENCE_THRESHOLD`.

For each seed, rules are selected using the training/validation rule-selection process, a threshold-calibration model is trained on training only, validation scores are used to convert the fixed quantile into bullish and bearish score thresholds, and a final model is refit on training plus validation before the frozen thresholds are applied once to the held-out test set.

In [21]:
ROBUST_THRESHOLD_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

robust_threshold_rows = []
robust_threshold_rule_rows = []
robust_threshold_prediction_frames = []

threshold_train = model_df.loc[model_df["period"].eq("training")].copy()
threshold_validation = model_df.loc[model_df["period"].eq("validation")].copy()
threshold_train_val = model_df.loc[model_df["period"].isin(["training", "validation"])].copy()
threshold_test = model_df.loc[model_df["period"].eq("test")].copy()

name = SELECTIVE_FILTER_VARIANT
base_spec = variant_specs[name]

for seed in ROBUST_THRESHOLD_SEEDS:
    print("selective-filter robustness seed", seed)

    candidate_rules, selected_rules = mine_validated_rules(
        threshold_train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    robust_threshold_rule_rows.append({
        "seed": seed,
        "variant": name,
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    robust_spec = base_spec.copy()
    robust_spec["selected_rules"] = selected_rules.copy()

    calibration_bundle = train_variant(
        f"{name}_threshold_calibration_seed_{seed}",
        robust_spec,
        threshold_train,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    validation_scored = score_variant(calibration_bundle, threshold_validation)

    bullish_threshold = float(
        validation_scored["score_bullish"].quantile(CONFIDENCE_THRESHOLD)
    )
    bearish_threshold = float(
        validation_scored["score_bearish"].quantile(CONFIDENCE_THRESHOLD)
    )

    final_bundle = train_variant(
        f"{name}_final_threshold_refit_seed_{seed}",
        robust_spec,
        threshold_train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    test_scored = score_variant(final_bundle, threshold_test)
    test_selective = apply_selective_thresholds(
        test_scored,
        bullish_threshold,
        bearish_threshold,
    )

    robust_threshold_prediction_frames.append(
        test_selective.assign(
            seed=seed,
            variant=name,
            period="test_repeated_seed_frozen_manual_thresholds",
            confidence_threshold_quantile=CONFIDENCE_THRESHOLD,
            bullish_threshold=bullish_threshold,
            bearish_threshold=bearish_threshold,
            n_selected_rules=len(selected_rules),
        )
    )

    for signal_label in ["bullish", "bearish", None]:
        row = selective_subset_stats(
            test_selective,
            variant=name,
            period="test_repeated_seed_frozen_manual_thresholds",
            signal_label=signal_label,
            threshold_quantile=CONFIDENCE_THRESHOLD,
        )
        row.update({
            "seed": seed,
            "bullish_threshold": bullish_threshold,
            "bearish_threshold": bearish_threshold,
            "n_selected_rules": len(selected_rules),
        })
        robust_threshold_rows.append(row)

robust_threshold_seed_summary = pd.DataFrame(robust_threshold_rows)
robust_threshold_rule_summary = pd.DataFrame(robust_threshold_rule_rows)
robust_threshold_predictions = pd.concat(
    robust_threshold_prediction_frames,
    ignore_index=True,
)

display(
    robust_threshold_seed_summary.sort_values(
        ["seed", "signal"]
    )
)

selective-filter robustness seed 1
selective-filter robustness seed 2
selective-filter robustness seed 3
selective-filter robustness seed 4
selective-filter robustness seed 5
selective-filter robustness seed 7
selective-filter robustness seed 11
selective-filter robustness seed 13
selective-filter robustness seed 17
selective-filter robustness seed 19


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct,seed,bullish_threshold,bearish_threshold,n_selected_rules
1,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bearish,0.85,58,11,18.965517,72.727273,0.715474,0.113281,-0.715474,1,0.999711,0.999957,21
0,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bullish,0.85,58,12,20.689655,41.666667,-1.200008,0.806152,-1.200008,1,0.999711,0.999957,21
2,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,joint,0.85,58,23,39.655172,56.521739,-0.283908,0.338820,-0.968274,1,0.999711,0.999957,21
4,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bearish,0.85,58,16,27.586207,68.750000,0.582557,0.105057,-0.582557,2,0.999999,0.999882,21
3,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bullish,0.85,58,7,12.068966,42.857143,-1.586855,0.773438,-1.586855,2,0.999999,0.999882,21
5,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,joint,0.85,58,23,39.655172,60.869565,-0.077699,0.202436,-0.888213,2,0.999999,0.999882,21
7,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bearish,0.85,58,22,37.931034,63.636364,0.123021,0.143139,-0.123021,3,0.999990,0.998033,21
6,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bullish,0.85,58,7,12.068966,57.142857,-0.858574,0.500000,-0.858574,3,0.999990,0.998033,21
8,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,joint,0.85,58,29,50.000000,62.068966,-0.113916,0.132465,-0.300568,3,0.999990,0.998033,21
10,selective_ltn_qwen_plus_context,test_repeated_seed_frozen_manual_thresholds,bearish,0.85,58,8,13.793103,87.500000,2.413828,0.035156,-2.413828,4,0.999993,0.999987,21


In [22]:
robust_threshold_summary = (
    robust_threshold_seed_summary
    .groupby(["variant", "signal"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        n_total=("n_total", "first"),
        n_signal_mean=("n_signal", "mean"),
        n_signal_min=("n_signal", "min"),
        n_signal_max=("n_signal", "max"),
        coverage_mean=("coverage_pct", "mean"),
        coverage_std=("coverage_pct", "std"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        accuracy_min=("accuracy_pct", "min"),
        accuracy_max=("accuracy_pct", "max"),
        aligned_mean_mean=("aligned_mean_pct", "mean"),
        aligned_mean_std=("aligned_mean_pct", "std"),
        p_value_median=("aligned_signflip_p", "median"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
)

display(
    robust_threshold_summary
    .sort_values(["signal"])
    .round({
        "n_signal_mean": 2,
        "coverage_mean": 2,
        "coverage_std": 2,
        "accuracy_mean": 2,
        "accuracy_std": 2,
        "accuracy_min": 2,
        "accuracy_max": 2,
        "aligned_mean_mean": 4,
        "aligned_mean_std": 4,
        "p_value_median": 4,
        "selected_rules_mean": 2,
    })
)

,variant,signal,n_seeds,n_total,n_signal_mean,n_signal_min,n_signal_max,coverage_mean,coverage_std,accuracy_mean,accuracy_std,accuracy_min,accuracy_max,aligned_mean_mean,aligned_mean_std,p_value_median,selected_rules_mean,selected_rules_min,selected_rules_max
0,selective_ltn_qwen_plus_context,bearish,10,58,14.9,8,22,25.69,7.06,69.53,7.85,57.14,87.50,0.7466,0.6716,0.1092,21.0,21,21
1,selective_ltn_qwen_plus_context,bullish,10,58,10.2,6,19,17.59,6.89,46.32,8.49,33.33,62.50,-0.8696,0.7309,0.7495,21.0,21,21
2,selective_ltn_qwen_plus_context,joint,10,58,25.1,20,33,43.28,6.63,59.44,4.73,51.52,67.86,0.0834,0.3850,0.2567,21.0,21,21


### 15.4 Selected Rule Diagnostics

This table inspects the selected rules used by the final `selective_ltn_qwen_plus_context` model. The rules were selected using training and validation data only; the held-out test columns below are post-hoc diagnostics to help interpret what kinds of event/context patterns the LTN relied on.

The test statistics should not be treated as a second rule-selection step. They are included only to check whether the selected rules have plausible behaviour on held-out events and to support qualitative interpretation of the model.

In [23]:
pd.set_option("display.max_colwidth", None)
def rule_condition_mask(data, condition):
    cols = [c.strip() for c in condition.split(" & ")]

    mask = pd.Series(True, index=data.index)
    for col in cols:
        if col not in data.columns:
            mask &= False
        else:
            mask &= pd.to_numeric(data[col], errors="coerce").fillna(0).ge(ANTECEDENT_THRESHOLD)

    return mask


rule_test_rows = []
selected_rules_for_test = frozen_variant_specs[SELECTIVE_FILTER_VARIANT]["selected_rules"].copy()
threshold_test = model_df.loc[model_df["period"].eq("test")].copy()

for _, rule in selected_rules_for_test.iterrows():
    mask = rule_condition_mask(threshold_test, rule["condition"])
    subset = threshold_test.loc[mask].copy()
    label = rule["rule_label"]

    if subset.empty:
        test_accuracy = np.nan
        test_aligned_mean = np.nan
        test_p = np.nan
    else:
        test_accuracy = float(100 * subset["target_label"].eq(label).mean())
        test_aligned_mean, test_p = label_return_alignment(
            subset,
            label,
            "simple_return",
        )

    rule_test_rows.append({
        "rule_label": label,
        "condition": rule["condition"],
        "rule_blocks": rule["rule_blocks"],
        "training_n": rule.get("training_n", np.nan),
        "training_accuracy_pct": rule.get("training_accuracy_pct", np.nan),
        "validation_n": rule.get("validation_n", np.nan),
        "validation_accuracy_pct": rule.get("validation_accuracy_pct", np.nan),
        "test_n": int(len(subset)),
        "test_accuracy_pct": test_accuracy,
        "test_aligned_mean_pct": test_aligned_mean,
        "test_aligned_signflip_p": test_p,
        "rule_score": rule["rule_score"],
    })

rule_test_summary = pd.DataFrame(rule_test_rows)

display(
    rule_test_summary
    .sort_values(
        ["test_accuracy_pct", "test_n", "validation_accuracy_pct"],
        ascending=False,
    )
    .head(20)
)

,rule_label,condition,rule_blocks,training_n,training_accuracy_pct,validation_n,validation_accuracy_pct,test_n,test_accuracy_pct,test_aligned_mean_pct,test_aligned_signflip_p,rule_score
0,bullish,novelty_new_event & similar_density_ticker_30d_01,"market_context, qwen_semantic",8,87.500000,11,90.909091,3,100.000000,2.902509,0.125000,3.350758
13,bearish,novelty_material_update & event_subtype_partnership_customer_deal,qwen_semantic,9,77.777778,8,75.000000,3,100.000000,2.932137,0.125000,2.525903
17,bearish,novel_context_30d & event_subtype_lawsuit_filed,"market_context, qwen_semantic",11,63.636364,6,66.666667,3,100.000000,1.051409,0.125000,2.180483
15,bearish,materiality_strength_medium & is_friday,"market_context, qwen_semantic",9,66.666667,7,71.428571,2,100.000000,NaN,NaN,2.742558
16,bearish,materiality_strength_medium & event_subtype_partnership_customer_deal,qwen_semantic,6,83.333333,7,71.428571,2,100.000000,NaN,NaN,2.635163
8,bearish,materiality_strength_medium & event_subtype_other,qwen_semantic,10,70.000000,5,80.000000,10,80.000000,1.249765,0.054688,2.916667
19,bearish,hour_cos_01 & event_subtype_other,"market_context, qwen_semantic",10,70.000000,5,60.000000,8,75.000000,1.267185,0.144531,2.716667
3,bullish,expected_label_bullish & valuation_channel_product_adoption,qwen_semantic,23,65.217391,12,83.333333,4,75.000000,3.917577,0.312500,3.460507
9,bearish,expected_label_neutral & event_subtype_other,qwen_semantic,10,60.000000,5,80.000000,9,66.666667,-0.009539,0.253906,2.715744
12,bullish,novelty_new_event & relative_prior_down_5d,"market_context, qwen_semantic",8,62.500000,8,75.000000,6,66.666667,2.866952,0.343750,2.784390


In [24]:
rule_test_summary[rule_test_summary["test_n"] > 8]

,rule_label,condition,rule_blocks,training_n,training_accuracy_pct,validation_n,validation_accuracy_pct,test_n,test_accuracy_pct,test_aligned_mean_pct,test_aligned_signflip_p,rule_score
8,bearish,materiality_strength_medium & event_subtype_other,qwen_semantic,10,70.0,5,80.0,10,80.000000,1.249765,0.054688,2.916667
9,bearish,expected_label_neutral & event_subtype_other,qwen_semantic,10,60.0,5,80.0,9,66.666667,-0.009539,0.253906,2.715744


## 16. Longer Held-Out Selective High-Confidence Threshold Filter

This section repeats the Section 15 selective-filter analysis on the longer held-out robustness split. It uses the same fixed `selective_ltn_qwen_plus_context` variant and the same manually chosen `CONFIDENCE_THRESHOLD`, but applies them to the longer test period.

The longer split is:

- training: `2023-06-01` to `2024-12-31`
- validation: `2025-01-01` to `2025-06-30`
- test: `2025-07-01` to `2026-06-01`

As in Section 15, validation is used for threshold calibration and the test set is used only after the threshold is frozen.

In [25]:
# Longer held-out selective high-confidence threshold setup.
#
# This section reuses the Section 15 helper functions:
# - apply_selective_thresholds
# - selective_subset_stats
# - choose_label_threshold
#
# It applies the same selective-filter logic to the longer robustness split.

LONGER_SELECTIVE_FILTER_VARIANT = SELECTIVE_FILTER_VARIANT

if LONGER_SELECTIVE_FILTER_VARIANT not in variant_specs:
    raise KeyError(
        f"{LONGER_SELECTIVE_FILTER_VARIANT!r} was not found in variant_specs. "
        f"Available variants: {list(variant_specs)}"
    )

required_longer_split_objects = [
    "model_df_robust",
    "train_val_robust",
    "test_robust",
]

missing_longer_split_objects = [
    name for name in required_longer_split_objects
    if name not in globals()
]

if missing_longer_split_objects:
    raise NameError(
        "Section 16 requires the longer held-out split objects from the earlier "
        f"robustness section. Missing: {missing_longer_split_objects}"
    )

longer_threshold_train = model_df_robust.loc[model_df_robust["period"].eq("training")].copy()
longer_threshold_validation = model_df_robust.loc[model_df_robust["period"].eq("validation")].copy()
longer_threshold_train_val = model_df_robust.loc[
    model_df_robust["period"].isin(["training", "validation"])
].copy()
longer_threshold_test = model_df_robust.loc[model_df_robust["period"].eq("test")].copy()

display(
    model_df_robust
    .groupby("period")["target_label"]
    .value_counts()
    .unstack(fill_value=0)
)

target_label,bearish,bullish
period,,
test,58,65
training,76,85
validation,32,29


### 16.1 Longer-Split Validation Threshold Quantile Sweep

This stage repeats the validation-only threshold sweep on the longer held-out robustness split. The longer-split training period is used to fit the threshold-calibration model, and the longer-split validation period is used to inspect the accuracy/coverage trade-off across candidate confidence threshold quantiles.

The longer held-out test period is not scored or inspected in this stage.

In [26]:
longer_threshold_bundles = {}
longer_threshold_rows = []
longer_threshold_search_frames = []
longer_threshold_grid_rows = []

name = LONGER_SELECTIVE_FILTER_VARIANT
base_spec = variant_specs[name]

candidate_rules, selected_rules = mine_validated_rules(
    longer_threshold_train_val,
    base_spec["feature_cols"],
    base_spec["return_col"],
    rule_blocks=base_spec.get("rule_blocks"),
    max_pair_features=base_spec.get("max_pair_features", 36),
    max_rules_per_label=base_spec.get("max_rules_per_label", 12),
)

longer_spec = base_spec.copy()
longer_spec["selected_rules"] = selected_rules.copy()

print(
    "longer selective threshold calibration",
    name,
    "training rows",
    len(longer_threshold_train),
)

longer_threshold_bundles[name] = train_variant(
    f"{name}_longer_threshold_calibration",
    longer_spec,
    longer_threshold_train,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

longer_validation_scored = score_variant(
    longer_threshold_bundles[name],
    longer_threshold_validation,
)

longer_bullish_choice, longer_bullish_search = choose_label_threshold(
    longer_validation_scored,
    "bullish",
)
longer_bearish_choice, longer_bearish_search = choose_label_threshold(
    longer_validation_scored,
    "bearish",
)

longer_bullish_threshold = float(longer_bullish_choice["threshold"])
longer_bearish_threshold = float(longer_bearish_choice["threshold"])

longer_threshold_rows.append({
    "variant": name,
    "bullish_quantile": float(longer_bullish_choice["quantile"]),
    "bullish_threshold": longer_bullish_threshold,
    "bullish_validation_n": int(longer_bullish_choice["n_signal"]),
    "bullish_validation_accuracy_pct": float(longer_bullish_choice["accuracy_pct"]),
    "bearish_quantile": float(longer_bearish_choice["quantile"]),
    "bearish_threshold": longer_bearish_threshold,
    "bearish_validation_n": int(longer_bearish_choice["n_signal"]),
    "bearish_validation_accuracy_pct": float(longer_bearish_choice["accuracy_pct"]),
    "n_candidate_rules": len(candidate_rules),
    "n_selected_rules": len(selected_rules),
})

longer_threshold_search_frames.append(
    pd.concat(
        [longer_bullish_search, longer_bearish_search],
        ignore_index=True,
    ).assign(variant=name)
)

for q in THRESHOLD_QUANTILES:
    bullish_q_threshold = float(longer_validation_scored["score_bullish"].quantile(q))
    bearish_q_threshold = float(longer_validation_scored["score_bearish"].quantile(q))

    validation_q = apply_selective_thresholds(
        longer_validation_scored,
        bullish_q_threshold,
        bearish_q_threshold,
    )

    for signal_label in ["bullish", "bearish", None]:
        longer_threshold_grid_rows.append(
            selective_subset_stats(
                validation_q,
                variant=name,
                period="longer_validation_threshold_grid",
                signal_label=signal_label,
                threshold_quantile=q,
            )
        )

longer_threshold_selection_summary = pd.DataFrame(longer_threshold_rows)
longer_threshold_search_results = pd.concat(
    longer_threshold_search_frames,
    ignore_index=True,
)
longer_threshold_grid_summary = pd.DataFrame(longer_threshold_grid_rows)

display(
    longer_threshold_grid_summary
    .sort_values(["threshold_quantile", "signal"])
    .loc[:, [
        "variant",
        "period",
        "signal",
        "threshold_quantile",
        "n_total",
        "n_signal",
        "coverage_pct",
        "accuracy_pct",
        "aligned_mean_pct",
        "aligned_signflip_p",
        "mean_simple_return_pct",
    ]]
)

longer selective threshold calibration selective_ltn_qwen_plus_context training rows 161


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct
1,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bearish,0.50,61,29,47.540984,48.275862,0.859249,0.644464,-0.859249
0,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bullish,0.50,61,29,47.540984,41.379310,-0.500217,0.867535,-0.500217
2,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,joint,0.50,61,58,95.081967,44.827586,0.179516,0.820928,-0.679733
4,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bearish,0.60,61,25,40.983607,44.000000,0.219467,0.787822,-0.219467
3,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bullish,0.60,61,25,40.983607,36.000000,-2.104616,0.946124,-2.104616
5,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,joint,0.60,61,50,81.967213,40.000000,-0.942574,0.940540,-1.162041
7,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bearish,0.70,61,19,31.147541,31.578947,-1.340327,0.968216,1.340327
6,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bullish,0.70,61,19,31.147541,36.842105,-1.468305,0.916466,-1.468305
8,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,joint,0.70,61,38,62.295082,34.210526,-1.404316,0.983224,-0.063989
10,selective_ltn_qwen_plus_context,longer_validation_threshold_grid,bearish,0.75,61,16,26.229508,31.250000,-1.311232,0.961594,1.311232


In [27]:
CONFIDENCE_THRESHOLD = 0.50

### 16.2 Longer Held-Out Frozen-Threshold Test

This stage freezes the confidence threshold quantile selected from the longer-split validation table. The final model is then refit on the longer-split training plus validation periods using the already frozen rules, and the frozen thresholds are applied once to the longer held-out test set.

The longer held-out test set is used only in this final evaluation stage.

In [28]:
longer_final_threshold_bundles = {}
longer_selective_test_rows = []
longer_selective_test_frames = []

name = LONGER_SELECTIVE_FILTER_VARIANT
spec = longer_spec

# Use the manually selected validation quantile on the longer-split validation scores.
longer_bullish_threshold = float(
    longer_validation_scored["score_bullish"].quantile(CONFIDENCE_THRESHOLD)
)
longer_bearish_threshold = float(
    longer_validation_scored["score_bearish"].quantile(CONFIDENCE_THRESHOLD)
)

longer_manual_threshold_selection_summary = pd.DataFrame([{
    "variant": name,
    "confidence_threshold_quantile": CONFIDENCE_THRESHOLD,
    "bullish_threshold": longer_bullish_threshold,
    "bearish_threshold": longer_bearish_threshold,
    "selection_mode": "manual_longer_validation_quantile",
    "n_selected_rules": len(selected_rules),
}])

# display(longer_manual_threshold_selection_summary)

print(
    "longer final selective-filter refit",
    name,
    "training+validation rows",
    len(longer_threshold_train_val),
)

longer_final_threshold_bundles[name] = train_variant(
    f"{name}_longer_final_threshold_refit",
    spec,
    longer_threshold_train_val,
    use_logic=True,
    logic_weight=LOGIC_WEIGHT,
    seed=SEED,
    mine_rules=False,
)

longer_test_scored = score_variant(
    longer_final_threshold_bundles[name],
    longer_threshold_test,
)
longer_test_selective = apply_selective_thresholds(
    longer_test_scored,
    longer_bullish_threshold,
    longer_bearish_threshold,
)

for signal_label in ["bullish", "bearish", None]:
    longer_selective_test_rows.append(
        selective_subset_stats(
            longer_test_selective,
            variant=name,
            period="longer_test_frozen_manual_validation_thresholds",
            signal_label=signal_label,
            threshold_quantile=CONFIDENCE_THRESHOLD,
        )
    )

longer_selective_test_frames.append(
    longer_test_selective.assign(
        variant=name,
        period="longer_test_frozen_manual_validation_thresholds",
        confidence_threshold_quantile=CONFIDENCE_THRESHOLD,
        bullish_threshold=longer_bullish_threshold,
        bearish_threshold=longer_bearish_threshold,
    )
)

longer_selective_high_confidence_summary = pd.DataFrame(longer_selective_test_rows)
longer_selective_high_confidence_predictions = pd.concat(
    longer_selective_test_frames,
    ignore_index=True,
)

display(
    longer_selective_high_confidence_summary.sort_values(
        ["signal", "accuracy_pct", "coverage_pct"],
        ascending=[True, False, False],
    )
)

longer final selective-filter refit selective_ltn_qwen_plus_context training+validation rows 222


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct
1,selective_ltn_qwen_plus_context,longer_test_frozen_manual_validation_thresholds,bearish,0.5,123,49,39.837398,53.061224,-0.751373,0.387725,0.751373
0,selective_ltn_qwen_plus_context,longer_test_frozen_manual_validation_thresholds,bullish,0.5,123,60,48.780488,56.666667,0.740548,0.183147,0.740548
2,selective_ltn_qwen_plus_context,longer_test_frozen_manual_validation_thresholds,joint,0.5,123,109,88.617886,55.045872,0.069868,0.169093,0.745414


### 16.3 Repeated-Seed Robustness On The Longer Held-Out Test

This robustness check repeats the longer-split selective-filter procedure across random seeds for the fixed `selective_ltn_qwen_plus_context` variant. The confidence threshold quantile is not re-selected per seed; each run uses the manually chosen `CONFIDENCE_THRESHOLD`.

For each seed, rules are selected using the longer-split training/validation rule-selection process, a threshold-calibration model is trained on the longer-split training period only, validation scores are used to convert the fixed quantile into bullish and bearish score thresholds, and a final model is refit on longer-split training plus validation before the frozen thresholds are applied once to the longer held-out test set.

In [29]:
LONGER_ROBUST_THRESHOLD_SEEDS = [1, 2, 3, 4, 5, 7, 11, 13, 17, 19]

longer_robust_threshold_rows = []
longer_robust_threshold_rule_rows = []
longer_robust_threshold_prediction_frames = []

name = LONGER_SELECTIVE_FILTER_VARIANT
base_spec = variant_specs[name]

for seed in LONGER_ROBUST_THRESHOLD_SEEDS:
    print("longer selective-filter robustness seed", seed)

    candidate_rules, selected_rules = mine_validated_rules(
        longer_threshold_train_val,
        base_spec["feature_cols"],
        base_spec["return_col"],
        rule_blocks=base_spec.get("rule_blocks"),
        max_pair_features=base_spec.get("max_pair_features", 36),
        max_rules_per_label=base_spec.get("max_rules_per_label", 12),
    )

    longer_robust_threshold_rule_rows.append({
        "seed": seed,
        "variant": name,
        "n_candidate_rules": len(candidate_rules),
        "n_selected_rules": len(selected_rules),
        "validation_n_min": selected_rules["validation_n"].min() if len(selected_rules) else np.nan,
        "validation_n_median": selected_rules["validation_n"].median() if len(selected_rules) else np.nan,
        "validation_n_max": selected_rules["validation_n"].max() if len(selected_rules) else np.nan,
        "validation_accuracy_mean": selected_rules["validation_accuracy_pct"].mean() if len(selected_rules) else np.nan,
    })

    robust_spec = base_spec.copy()
    robust_spec["selected_rules"] = selected_rules.copy()

    calibration_bundle = train_variant(
        f"{name}_longer_threshold_calibration_seed_{seed}",
        robust_spec,
        longer_threshold_train,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    validation_scored = score_variant(calibration_bundle, longer_threshold_validation)

    bullish_threshold = float(
        validation_scored["score_bullish"].quantile(CONFIDENCE_THRESHOLD)
    )
    bearish_threshold = float(
        validation_scored["score_bearish"].quantile(CONFIDENCE_THRESHOLD)
    )

    final_bundle = train_variant(
        f"{name}_longer_final_threshold_refit_seed_{seed}",
        robust_spec,
        longer_threshold_train_val,
        use_logic=True,
        logic_weight=LOGIC_WEIGHT,
        seed=seed,
        mine_rules=False,
    )

    test_scored = score_variant(final_bundle, longer_threshold_test)
    test_selective = apply_selective_thresholds(
        test_scored,
        bullish_threshold,
        bearish_threshold,
    )

    longer_robust_threshold_prediction_frames.append(
        test_selective.assign(
            seed=seed,
            variant=name,
            period="longer_test_repeated_seed_frozen_manual_thresholds",
            confidence_threshold_quantile=CONFIDENCE_THRESHOLD,
            bullish_threshold=bullish_threshold,
            bearish_threshold=bearish_threshold,
            n_selected_rules=len(selected_rules),
        )
    )

    for signal_label in ["bullish", "bearish", None]:
        row = selective_subset_stats(
            test_selective,
            variant=name,
            period="longer_test_repeated_seed_frozen_manual_thresholds",
            signal_label=signal_label,
            threshold_quantile=CONFIDENCE_THRESHOLD,
        )
        row.update({
            "seed": seed,
            "bullish_threshold": bullish_threshold,
            "bearish_threshold": bearish_threshold,
            "n_selected_rules": len(selected_rules),
        })
        longer_robust_threshold_rows.append(row)

longer_robust_threshold_seed_summary = pd.DataFrame(longer_robust_threshold_rows)
longer_robust_threshold_rule_summary = pd.DataFrame(longer_robust_threshold_rule_rows)
longer_robust_threshold_predictions = pd.concat(
    longer_robust_threshold_prediction_frames,
    ignore_index=True,
)

display(
    longer_robust_threshold_seed_summary.sort_values(
        ["seed", "signal"]
    )
)

longer selective-filter robustness seed 1
longer selective-filter robustness seed 2
longer selective-filter robustness seed 3
longer selective-filter robustness seed 4
longer selective-filter robustness seed 5
longer selective-filter robustness seed 7
longer selective-filter robustness seed 11
longer selective-filter robustness seed 13
longer selective-filter robustness seed 17
longer selective-filter robustness seed 19


,variant,period,signal,threshold_quantile,n_total,n_signal,coverage_pct,accuracy_pct,aligned_mean_pct,aligned_signflip_p,mean_simple_return_pct,seed,bullish_threshold,bearish_threshold,n_selected_rules
1,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bearish,0.5,123,53,43.089431,52.830189,-0.780912,0.391923,0.780912,1,0.383696,0.034063,20
0,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bullish,0.5,123,58,47.154472,63.793103,1.263031,0.023970,1.263031,1,0.383696,0.034063,20
2,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,joint,0.5,123,111,90.243902,58.558559,0.287094,0.043545,1.032830,1,0.383696,0.034063,20
4,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bearish,0.5,123,41,33.333333,56.097561,-0.352886,0.266355,0.352886,2,0.590884,0.953645,20
3,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bullish,0.5,123,56,45.528455,51.785714,0.469190,0.446927,0.469190,2,0.590884,0.953645,20
5,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,joint,0.5,123,97,78.861789,53.608247,0.121714,0.271305,0.420030,2,0.590884,0.953645,20
7,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bearish,0.5,123,66,53.658537,53.030303,-1.019198,0.356115,1.019198,3,0.910586,0.007074,20
6,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bullish,0.5,123,35,28.455285,60.000000,0.664578,0.155252,0.664578,3,0.910586,0.007074,20
8,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,joint,0.5,123,101,82.113821,55.445545,-0.435711,0.159864,0.896310,3,0.910586,0.007074,20
10,selective_ltn_qwen_plus_context,longer_test_repeated_seed_frozen_manual_thresholds,bearish,0.5,123,53,43.089431,58.490566,-0.207276,0.135840,0.207276,4,0.765967,0.632520,20


In [30]:
longer_robust_threshold_summary = (
    longer_robust_threshold_seed_summary
    .groupby(["variant", "signal"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        n_total=("n_total", "first"),
        n_signal_mean=("n_signal", "mean"),
        n_signal_min=("n_signal", "min"),
        n_signal_max=("n_signal", "max"),
        coverage_mean=("coverage_pct", "mean"),
        coverage_std=("coverage_pct", "std"),
        accuracy_mean=("accuracy_pct", "mean"),
        accuracy_std=("accuracy_pct", "std"),
        accuracy_min=("accuracy_pct", "min"),
        accuracy_max=("accuracy_pct", "max"),
        aligned_mean_mean=("aligned_mean_pct", "mean"),
        aligned_mean_std=("aligned_mean_pct", "std"),
        p_value_median=("aligned_signflip_p", "median"),
        selected_rules_mean=("n_selected_rules", "mean"),
        selected_rules_min=("n_selected_rules", "min"),
        selected_rules_max=("n_selected_rules", "max"),
    )
)

display(
    longer_robust_threshold_summary
    .sort_values(["signal"])
    .round({
        "n_signal_mean": 2,
        "coverage_mean": 2,
        "coverage_std": 2,
        "accuracy_mean": 2,
        "accuracy_std": 2,
        "accuracy_min": 2,
        "accuracy_max": 2,
        "aligned_mean_mean": 4,
        "aligned_mean_std": 4,
        "p_value_median": 4,
        "selected_rules_mean": 2,
    })
)

,variant,signal,n_seeds,n_total,n_signal_mean,n_signal_min,n_signal_max,coverage_mean,coverage_std,accuracy_mean,accuracy_std,accuracy_min,accuracy_max,aligned_mean_mean,aligned_mean_std,p_value_median,selected_rules_mean,selected_rules_min,selected_rules_max
0,selective_ltn_qwen_plus_context,bearish,10,123,51.6,41,66,41.95,5.58,56.03,3.35,52.83,63.64,-0.5021,0.3327,0.2790,20.0,20,20
1,selective_ltn_qwen_plus_context,bullish,10,123,54.2,35,60,44.07,6.03,59.30,3.62,51.79,64.00,0.9134,0.2828,0.1292,20.0,20,20
2,selective_ltn_qwen_plus_context,joint,10,123,105.8,94,113,86.02,5.15,57.60,2.99,53.61,63.83,0.2149,0.3250,0.0918,20.0,20,20


### 16.4 Longer Held-Out Selected Rule Diagnostics

This table inspects the selected rules used by the longer-split `selective_ltn_qwen_plus_context` model. The rules were selected using the longer-split training and validation periods only; the longer held-out test columns are post-hoc diagnostics to help interpret what kinds of event/context patterns the LTN relied on.

The longer-test statistics should not be treated as a second rule-selection step. They are included only to check whether the selected rules have plausible behaviour on the longer held-out period and to support qualitative interpretation of the model.

In [31]:
pd.set_option("display.max_colwidth", None)

longer_rule_test_rows = []
selected_longer_rules_for_test = longer_spec["selected_rules"].copy()

for _, rule in selected_longer_rules_for_test.iterrows():
    mask = rule_condition_mask(longer_threshold_test, rule["condition"])
    subset = longer_threshold_test.loc[mask].copy()
    label = rule["rule_label"]

    if subset.empty:
        test_accuracy = np.nan
        test_aligned_mean = np.nan
        test_p = np.nan
    else:
        test_accuracy = float(100 * subset["target_label"].eq(label).mean())
        test_aligned_mean, test_p = label_return_alignment(
            subset,
            label,
            "simple_return",
        )

    longer_rule_test_rows.append({
        "rule_label": label,
        "condition": rule["condition"],
        "rule_blocks": rule["rule_blocks"],
        "training_n": rule.get("training_n", np.nan),
        "training_accuracy_pct": rule.get("training_accuracy_pct", np.nan),
        "validation_n": rule.get("validation_n", np.nan),
        "validation_accuracy_pct": rule.get("validation_accuracy_pct", np.nan),
        "longer_test_n": int(len(subset)),
        "longer_test_accuracy_pct": test_accuracy,
        "longer_test_aligned_mean_pct": test_aligned_mean,
        "longer_test_aligned_signflip_p": test_p,
        "rule_score": rule["rule_score"],
    })

longer_rule_test_summary = pd.DataFrame(longer_rule_test_rows)

display(
    longer_rule_test_summary
    .sort_values(
        ["longer_test_accuracy_pct", "longer_test_n", "validation_accuracy_pct"],
        ascending=False,
    )
    .head(20)
)

,rule_label,condition,rule_blocks,training_n,training_accuracy_pct,validation_n,validation_accuracy_pct,longer_test_n,longer_test_accuracy_pct,longer_test_aligned_mean_pct,longer_test_aligned_signflip_p,rule_score
4,bullish,novelty_new_event & similar_density_ticker_30d_01,"market_context, qwen_semantic",8,87.500000,6,83.333333,8,100.000000,3.904502,0.003906,3.108333
14,bearish,materiality_strength_medium & event_subtype_partnership_customer_deal,qwen_semantic,6,83.333333,6,66.666667,3,100.000000,1.821653,0.125000,2.554211
2,bullish,expected_label_bullish & valuation_channel_product_adoption,qwen_semantic,23,65.217391,6,83.333333,10,80.000000,3.478214,0.054688,3.260507
1,bullish,include_candidate & valuation_channel_product_adoption,qwen_semantic,23,65.217391,6,83.333333,11,72.727273,2.776453,0.113281,3.260507
3,bullish,materiality_strength_high & valuation_channel_product_adoption,qwen_semantic,19,73.684211,6,83.333333,11,72.727273,2.296887,0.113281,3.245175
6,bullish,materiality_strength_high & similar_density_ticker_30d_01,"market_context, qwen_semantic",15,60.000000,9,77.777778,22,68.181818,1.687967,0.066900,2.726778
19,bearish,expected_label_bullish & materiality_strength_medium,qwen_semantic,10,70.000000,5,60.000000,3,66.666667,0.807797,0.500000,1.890820
8,bullish,valuation_channel_product_adoption,qwen_semantic,26,61.538462,7,71.428571,13,61.538462,1.447662,0.290527,3.213004
9,bullish,article_count & valuation_channel_product_adoption,qwen_semantic,26,61.538462,7,71.428571,13,61.538462,1.447662,0.290527,3.213004
0,bullish,novelty_new_event & valuation_channel_product_adoption,qwen_semantic,21,76.190476,6,83.333333,10,60.000000,0.939266,0.376953,3.320238
